In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



# new lag features with baseling rolling features

In [3]:
df = pd.read_csv("/kaggle/input/datasets/logeshm0324/df-for-eda/df_for_EDA.csv")

In [4]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,28570.0,27669.0,26498.0,29475.375000,30894.589286,2736.646326
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,26162.0,25147.0,29453.750000,30897.541667,2765.861628
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,25483.0,24574.0,29429.750000,30899.523810,2804.050165
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,25045.0,24393.0,29416.250000,30901.476190,2826.766154
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,25030.0,24860.0,29421.000000,30903.166667,2819.160745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,45787.0,42112.0,40343.500000,41856.327381,2295.270146
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,45209.0,40797.0,40282.750000,41873.910714,2177.148998
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,43663.0,38819.0,40230.208333,41895.238095,2106.081917
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,41581.0,36287.0,40171.166667,41918.315476,2086.336995


In [5]:
df["Datetime"] = pd.to_datetime(df["Datetime"])


df["Hour"] = df["Datetime"].dt.hour

df["Day"] = df["Datetime"].dt.day

df["DayOfWeek"] = df["Datetime"].dt.dayofweek

df["Week"] = df["Datetime"].dt.isocalendar().week.astype(int)

df["Month"] = df["Datetime"].dt.month

df["Year"] = df["Datetime"].dt.year

df["IsWeekend"] = ( df["DayOfWeek"] >= 5 ).astype(int)

# Lag 

df["Lag_1"] = df["PJME_MW"].shift(1)
df["Lag_2"] = df["PJME_MW"].shift(2)
df["Lag_3"] = df["PJME_MW"].shift(3)
df["Lag_6"] = df["PJME_MW"].shift(6)
df["Lag_12"] = df["PJME_MW"].shift(12)
df["Lag_24"] = df["PJME_MW"].shift(24)
df["Lag_48"] = df["PJME_MW"].shift(48)
df["Lag_72"] = df["PJME_MW"].shift(72)
df["Lag_168"] = df["PJME_MW"].shift(168)

# Rolling mean and std

df["RollingMean_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .mean()
)


df["RollingMean_168"] = (
    df["PJME_MW"]
    .rolling(168)
    .mean()
)


df["RollingStd_24"] = (
    df["PJME_MW"]
    .rolling(24)
    .std()
)

In [6]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-24 01:00:00,27213.0,1,3,12,24,52,1998,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1998-12-24 02:00:00,25643.0,2,3,12,24,52,1998,0,27213.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1998-12-24 03:00:00,24907.0,3,3,12,24,52,1998,0,25643.0,...,NaN,NaN,NaN,NaN,27213.0,NaN,NaN,NaN,NaN,NaN
3,1998-12-24 04:00:00,24721.0,4,3,12,24,52,1998,0,24907.0,...,NaN,NaN,NaN,NaN,25643.0,27213.0,NaN,NaN,NaN,NaN
4,1998-12-24 05:00:00,25144.0,5,3,12,24,52,1998,0,24721.0,...,NaN,NaN,NaN,NaN,24907.0,25643.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145193,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145194,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145195,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145196,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [7]:
df.isna().sum()

Datetime             0
PJME_MW              0
Hour                 0
DayOfWeek            0
Month                0
Day                  0
Week                 0
Year                 0
IsWeekend            0
Lag_1                1
Lag_24              24
Lag_168            168
RollingMean_24      23
RollingMean_168    167
RollingStd_24       23
Lag_2                2
Lag_3                3
Lag_6                6
Lag_12              12
Lag_48              48
Lag_72              72
dtype: int64

In [8]:
missing_count = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
)

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

missing_summary

,Missing Count,Missing Percentage
Datetime,0,0.000000
PJME_MW,0,0.000000
Hour,0,0.000000
DayOfWeek,0,0.000000
Month,0,0.000000
Day,0,0.000000
Week,0,0.000000
Year,0,0.000000
IsWeekend,0,0.000000
Lag_1,1,0.000689


In [9]:
df = df.dropna().reset_index(drop=True)

In [ ]:
df.to_csv("../Dataset/final_df.csv")

In [15]:
df

,Datetime,PJME_MW,Hour,DayOfWeek,Month,Day,Week,Year,IsWeekend,Lag_1,...,Lag_168,RollingMean_24,RollingMean_168,RollingStd_24,Lag_2,Lag_3,Lag_6,Lag_12,Lag_48,Lag_72
0,1998-12-17 01:00:00,29971.0,1,3,12,17,51,1998,0,32323.0,...,27213.0,35721.375000,31485.982143,3667.762970,35430.0,38234.0,40810.0,35867.0,29752.0,26437.0
1,1998-12-17 02:00:00,29046.0,2,3,12,17,51,1998,0,29971.0,...,25643.0,35676.666667,31506.238095,3744.754089,32323.0,35430.0,40317.0,35318.0,28489.0,24978.0
2,1998-12-17 03:00:00,28653.0,3,3,12,17,51,1998,0,29046.0,...,24907.0,35625.958333,31528.535714,3833.978618,29971.0,32323.0,39738.0,34817.0,27886.0,24353.0
3,1998-12-17 04:00:00,28774.0,4,3,12,17,51,1998,0,28653.0,...,24721.0,35573.875000,31552.660714,3920.893358,29046.0,29971.0,38234.0,34906.0,27712.0,24029.0
4,1998-12-17 05:00:00,29453.0,5,3,12,17,51,1998,0,28774.0,...,25144.0,35520.541667,31578.309524,3997.559484,28653.0,29046.0,35430.0,36661.0,28217.0,24363.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145025,2014-03-02 03:00:00,44284.0,3,6,3,2,9,2014,1,44343.0,...,42112.0,40343.500000,41856.327381,2295.270146,44147.0,41213.0,38726.0,40154.0,43308.0,45896.0
145026,2014-03-02 04:00:00,43751.0,4,6,3,2,9,2014,1,44284.0,...,40797.0,40282.750000,41873.910714,2177.148998,44343.0,44147.0,38737.0,40309.0,42440.0,45377.0
145027,2014-03-02 05:00:00,42402.0,5,6,3,2,9,2014,1,43751.0,...,38819.0,40230.208333,41895.238095,2106.081917,44284.0,44343.0,39337.0,39884.0,40661.0,44092.0
145028,2014-03-02 06:00:00,40164.0,6,6,3,2,9,2014,1,42402.0,...,36287.0,40171.166667,41918.315476,2086.336995,43751.0,44284.0,41213.0,39544.0,38207.0,42257.0


In [11]:
df.isna().sum()

Datetime           0
PJME_MW            0
Hour               0
DayOfWeek          0
Month              0
Day                0
Week               0
Year               0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64

In [10]:
features = [
    "PJME_MW",
    "Hour",
    "Day",
    "Week",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168",
    "RollingStd_24",
    "Lag_2",
    "Lag_3",
    "Lag_6",
    "Lag_12",
    "Lag_48",
    "Lag_72"
]

target = "PJME_MW"

X = df[features]
y = df[target]

In [11]:
train_size = int(len(df) * 0.70)
val_size = int(len(df) * 0.15)

X_train = X.iloc[:train_size]
X_val = X.iloc[train_size:train_size + val_size]
X_test = X.iloc[train_size + val_size:]

y_train = y.iloc[:train_size]
y_val = y.iloc[train_size:train_size + val_size]
y_test = y.iloc[train_size + val_size:]

In [12]:
from sklearn.preprocessing import StandardScaler

X_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(X_train)

X_val_scaled = X_scaler.transform(X_val)

X_test_scaled = X_scaler.transform(X_test)

In [ ]:
import joblib

joblib.dump(
    X_scaler,
    "../Models/X_scaler.pkl"
)

In [14]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train.values.reshape(-1, 1)
)

y_val_scaled = y_scaler.transform(
    y_val.values.reshape(-1, 1)
)

y_test_scaled = y_scaler.transform(
    y_test.values.reshape(-1, 1)
)

In [16]:
import joblib

joblib.dump(
    y_scaler,
    "../Models/y_scaler.pkl"
)

['../Models/y_scaler.pkl']

In [15]:
print(X_train.isna().sum())
print(X_val.isna().sum())
print(X_test.isna().sum())

PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64
PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
RollingStd_24      0
Lag_2              0
Lag_3              0
Lag_6              0
Lag_12             0
Lag_48             0
Lag_72             0
dtype: int64
PJME_MW            0
Hour               0
Day                0
Week               0
Month              0
DayOfWeek          0
IsWeekend          0
Lag_1              0
Lag_24  

In [17]:
print("X_train inf:", np.isinf(X_train).sum().sum())
print("X_val inf:", np.isinf(X_val).sum().sum())
print("X_test inf:", np.isinf(X_test).sum().sum())

X_train inf: 0
X_val inf: 0
X_test inf: 0


In [18]:
print("X_train_scaled NaN:",
      np.isnan(X_train_scaled).sum())

print("X_val_scaled NaN:",
      np.isnan(X_val_scaled).sum())

print("X_test_scaled NaN:",
      np.isnan(X_test_scaled).sum())

X_train_scaled NaN: 0
X_val_scaled NaN: 0
X_test_scaled NaN: 0


In [19]:
print("X_train_scaled inf:",
      np.isinf(X_train_scaled).sum())

print("X_val_scaled inf:",
      np.isinf(X_val_scaled).sum())

print("X_test_scaled inf:",
      np.isinf(X_test_scaled).sum())

X_train_scaled inf: 0
X_val_scaled inf: 0
X_test_scaled inf: 0


In [16]:
def create_sequences(X, y, sequence_length, forecast_horizon):

    X_sequences = []
    y_sequences = []

    for i in range(
        sequence_length,
        len(X) - forecast_horizon + 1
    ):

        X_sequences.append(
            X[i-sequence_length:i]
        )

        y_sequences.append(
            y[i:i+forecast_horizon]
        )

    return np.array(X_sequences), np.array(y_sequences)

In [17]:
sequence_length = 48
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [77]:
print("X_train_seq NaN:",
      np.isnan(X_train_seq).sum())

print("y_train_seq NaN:",
      np.isnan(y_train_seq).sum())

print("X_val_seq NaN:",
      np.isnan(X_val_seq).sum())

print("y_val_seq NaN:",
      np.isnan(y_val_seq).sum())

X_train_seq NaN: 0
y_train_seq NaN: 0
X_val_seq NaN: 0
y_val_seq NaN: 0


In [78]:
X_train_seq[[1]]

array([[[-5.15692045e-01, -1.37279530e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -3.73941864e-01, -3.51688443e-01,
         -1.03819985e+00,  7.23487081e-01, -2.30702218e-01,
         -4.00556596e-01, -1.34839168e-02,  4.62690110e-01,
          1.21161940e+00,  4.45382318e-01, -6.01707547e-01,
         -1.13991387e+00],
        [-5.75923910e-01, -1.22832131e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -5.15710300e-01, -3.89855400e-01,
         -1.15105950e+00,  7.12249474e-01, -2.25027655e-01,
         -3.50956704e-01, -3.73961331e-01, -1.35019938e-02,
          1.12288080e+00,  3.68597573e-01, -6.94141272e-01,
         -1.23572709e+00],
        [-5.57379239e-01, -1.08384733e+00,  1.44294962e-01,
          1.60901014e+00,  1.57837541e+00,  5.86346421e-04,
         -6.31875576e-01, -5.75942728e-01, -3.66250133e-01,
         -1.17958110e+00,  7.00707149e-01, -2.

In [79]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [ ]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

In [81]:
bilstm_pred_48 = bilstm_model_48.predict(
    X_test_seq
)

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [82]:
bilstm_pred_48

array([[-0.01916628, -0.11162368, -0.1029131 , ..., -0.6244201 ,
        -0.55649674, -0.5953048 ],
       [-0.11098327, -0.14601988,  0.01467767, ..., -0.5620045 ,
        -0.5277519 , -0.6109145 ],
       [-0.03446794,  0.08518342,  0.33336633, ..., -0.54374725,
        -0.5576612 , -0.624085  ],
       ...,
       [ 1.4773215 ,  1.328861  ,  1.1779613 , ...,  2.5358148 ,
         2.3340333 ,  2.2212677 ],
       [ 1.16952   ,  1.0873945 ,  1.0776851 , ...,  2.4111333 ,
         2.0913062 ,  1.8290689 ],
       [ 1.0199214 ,  0.9878604 ,  1.0500486 , ...,  2.1658134 ,
         1.7268264 ,  1.416622  ]], dtype=float32)

# Metric Calculation block

In [19]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    
    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [84]:
bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48.reshape(-1, 1)
).reshape(bilstm_pred_48.shape)

In [18]:
y_test_original = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

In [87]:
bilstm_pred_original_48

array([[32285.725, 31682.459, 31739.295, ..., 28336.572, 28779.758,
        28526.543],
       [31686.639, 31458.031, 32506.549, ..., 28743.82 , 28967.312,
        28424.693],
       [32185.885, 32966.582, 34585.92 , ..., 28862.945, 28772.16 ,
        28338.76 ],
       ...,
       [42049.984, 41081.312, 40096.727, ..., 48956.43 , 47639.848,
        46904.074],
       [40041.65 , 39505.797, 39442.445, ..., 48142.906, 46056.105,
        44345.062],
       [39065.547, 38856.355, 39262.12 , ..., 46542.25 , 43677.953,
        41653.934]], dtype=float32)

In [89]:
results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [90]:
results

{'MAE': 1983.9874887875158,
 'MSE': 6996736.607804939,
 'RMSE': np.float64(2645.1345160133046),
 'MAPE': 6.674865726195193,
 'R2': 0.8323657998774167,
 'Bias': np.float64(939.277266175341)}

# Adding Drop out

In [93]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.2),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [94]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.1179 - mae: 0.2520 - val_loss: 0.0854 - val_mae: 0.2214
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0683 - mae: 0.1970 - val_loss: 0.0782 - val_mae: 0.2113
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0600 - mae: 0.1841 - val_loss: 0.0816 - val_mae: 0.2182
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0550 - mae: 0.1764 - val_loss: 0.0746 - val_mae: 0.2012
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0512 - mae: 0.1703 - val_loss: 0.0743 - val_mae: 0.2032
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0480 - mae: 0.1652 - val_loss: 0.0785 - val_mae: 0.2086
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0457 - mae: 0.1613 - val_loss: 0.0807 - val_mae: 0.2128
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0435 - mae: 0.1576 - val_loss: 0.0734 - val_mae: 0.2004
Epoch 9/10
1586/1586 ━━━━━━━━━━━

In [95]:
bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

678/678 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [96]:
bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

In [97]:
results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [98]:
results

{'MAE': 1942.9533460671366,
 'MSE': 7083305.258758675,
 'RMSE': np.float64(2661.4479628124755),
 'MAPE': 6.408207644955419,
 'R2': 0.8302917091445837,
 'Bias': np.float64(790.6286425787255)}

In [99]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.3),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [100]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 17s 9ms/step - loss: 0.1289 - mae: 0.2668 - val_loss: 0.0999 - val_mae: 0.2423
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0757 - mae: 0.2083 - val_loss: 0.0868 - val_mae: 0.2226
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0668 - mae: 0.1949 - val_loss: 0.0810 - val_mae: 0.2146
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0617 - mae: 0.1873 - val_loss: 0.0805 - val_mae: 0.2117
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0580 - mae: 0.1818 - val_loss: 0.0736 - val_mae: 0.2004
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0549 - mae: 0.1770 - val_loss: 0.0788 - val_mae: 0.2075
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0524 - mae: 0.1731 - val_loss: 0.0851 - val_mae: 0.2191
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0502 - mae: 0.1697 - val_loss: 0.0792 - val_mae: 0.2072
Epoch 9/10
1586/1586 ━━━━━━━━━━━

In [101]:
bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 1946.8032687056484,
 'MSE': 6910368.552649346,
 'RMSE': np.float64(2628.757986701961),
 'MAPE': 6.444103102086086,
 'R2': 0.8344350845530754,
 'Bias': np.float64(596.8464515947338)}

In [102]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

         tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [103]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_dp = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_dp.reshape(-1, 1)
).reshape(bilstm_pred_48_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1575 - mae: 0.2970 - val_loss: 0.0953 - val_mae: 0.2368
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0938 - mae: 0.2339 - val_loss: 0.0852 - val_mae: 0.2202
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0837 - mae: 0.2203 - val_loss: 0.0843 - val_mae: 0.2188
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0788 - mae: 0.2136 - val_loss: 0.0794 - val_mae: 0.2115
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0748 - mae: 0.2080 - val_loss: 0.0786 - val_mae: 0.2104
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0717 - mae: 0.2038 - val_loss: 0.0754 - val_mae: 0.2044
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0693 - mae: 0.2002 - val_loss: 0.0864 - val_mae: 0.2229
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0667 - mae: 0.1964 - val_loss: 0.0761 - val_mae: 0.2082
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 1824.9897190871102,
 'MSE': 6269557.12628878,
 'RMSE': np.float64(2503.9083701862533),
 'MAPE': 5.980970541785555,
 'R2': 0.8497882294417274,
 'Bias': np.float64(360.0020746323614)}

# Adding Batch Normaliation

In [104]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [105]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_bn = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bn.reshape(-1, 1)
).reshape(bilstm_pred_48_bn.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.2508 - mae: 0.3692 - val_loss: 0.1058 - val_mae: 0.2486
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1172 - mae: 0.2631 - val_loss: 0.1000 - val_mae: 0.2440
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1012 - mae: 0.2435 - val_loss: 0.0880 - val_mae: 0.2273
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0926 - mae: 0.2326 - val_loss: 0.0924 - val_mae: 0.2364
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0875 - mae: 0.2258 - val_loss: 0.0932 - val_mae: 0.2357
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0827 - mae: 0.2193 - val_loss: 0.0824 - val_mae: 0.2180
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0786 - mae: 0.2140 - val_loss: 0.0854 - val_mae: 0.2241
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0750 - mae: 0.2090 - val_loss: 0.0933 - val_mae: 0.2364
Epoch 9/10
1586/1586 ━━━━━━━━

{'MAE': 2190.5832167536705,
 'MSE': 8063596.924461169,
 'RMSE': np.float64(2839.6473239578836),
 'MAPE': 7.38142629382956,
 'R2': 0.8068049869084544,
 'Bias': np.float64(1121.0317592042652)}

# New Optimizers

In [107]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [108]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_op = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_op.reshape(-1, 1)
).reshape(bilstm_pred_48_op.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 1.0120 - mae: 0.7823 - val_loss: 0.7873 - val_mae: 0.7122
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.8471 - mae: 0.7169 - val_loss: 0.6788 - val_mae: 0.6583
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.7285 - mae: 0.6665 - val_loss: 0.5826 - val_mae: 0.6062
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.6332 - mae: 0.6232 - val_loss: 0.5003 - val_mae: 0.5595
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.5558 - mae: 0.5856 - val_loss: 0.4345 - val_mae: 0.5213
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.4912 - mae: 0.5519 - val_loss: 0.3826 - val_mae: 0.4900
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.4370 - mae: 0.5213 - val_loss: 0.3417 - val_mae: 0.4640
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.3937 - mae: 0.4953 - val_loss: 0.3095 - val_mae: 0.4421
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2922.8521461639284,
 'MSE': 13810281.711359901,
 'RMSE': np.float64(3716.218738362948),
 'MAPE': 9.818812926488706,
 'R2': 0.6691206689746094,
 'Bias': np.float64(353.546182138574)}

In [109]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_op = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_op.reshape(-1, 1)
).reshape(bilstm_pred_48_op.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1564 - mae: 0.2979 - val_loss: 0.0912 - val_mae: 0.2272
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0983 - mae: 0.2390 - val_loss: 0.0926 - val_mae: 0.2274
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0882 - mae: 0.2259 - val_loss: 0.0829 - val_mae: 0.2133
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0821 - mae: 0.2179 - val_loss: 0.0851 - val_mae: 0.2226
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0778 - mae: 0.2119 - val_loss: 0.0882 - val_mae: 0.2274
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0748 - mae: 0.2076 - val_loss: 0.0819 - val_mae: 0.2172
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0723 - mae: 0.2040 - val_loss: 0.0914 - val_mae: 0.2299
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0703 - mae: 0.2012 - val_loss: 0.0862 - val_mae: 0.2275
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 1771.425878130128,
 'MSE': 5619109.234146657,
 'RMSE': np.float64(2370.4660373324605),
 'MAPE': 5.793036501472333,
 'R2': 0.8653722535707811,
 'Bias': np.float64(72.40864041789405)}

In [110]:
bilstm_model_48.save("/kaggle/working/bilstm_model_48_RMS_86.keras")

# Changing Learning Rate

In [111]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.1556 - mae: 0.2966 - val_loss: 0.1123 - val_mae: 0.2528
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1102 - mae: 0.2524 - val_loss: 0.1045 - val_mae: 0.2426
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0988 - mae: 0.2391 - val_loss: 0.1062 - val_mae: 0.2472
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0925 - mae: 0.2314 - val_loss: 0.0992 - val_mae: 0.2394
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0882 - mae: 0.2260 - val_loss: 0.1193 - val_mae: 0.2707
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0843 - mae: 0.2208 - val_loss: 0.0991 - val_mae: 0.2400
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0815 - mae: 0.2172 - val_loss: 0.1275 - val_mae: 0.2852
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0796 - mae: 0.2148 - val_loss: 0.0900 - val_mae: 0.2260
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2130.885912750419,
 'MSE': 8005726.946277911,
 'RMSE': np.float64(2829.4393342635763),
 'MAPE': 6.992067660274932,
 'R2': 0.8081914886517158,
 'Bias': np.float64(446.30317291764544)}

In [112]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.005),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - loss: 0.1398 - mae: 0.2822 - val_loss: 0.1102 - val_mae: 0.2516
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0981 - mae: 0.2378 - val_loss: 0.0976 - val_mae: 0.2358
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0869 - mae: 0.2236 - val_loss: 0.1156 - val_mae: 0.2672
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0813 - mae: 0.2161 - val_loss: 0.0969 - val_mae: 0.2428
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0770 - mae: 0.2104 - val_loss: 0.0912 - val_mae: 0.2336
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0734 - mae: 0.2056 - val_loss: 0.0857 - val_mae: 0.2190
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0712 - mae: 0.2027 - val_loss: 0.0868 - val_mae: 0.2243
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0691 - mae: 0.1999 - val_loss: 0.0853 - val_mae: 0.2215
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2082.0503937970416,
 'MSE': 7367050.406823056,
 'RMSE': np.float64(2714.231089429022),
 'MAPE': 7.015326774344978,
 'R2': 0.8234934839718115,
 'Bias': np.float64(767.3909411308093)}

In [113]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.00025),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

bilstm_pred_48_lr = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_lr.reshape(-1, 1)
).reshape(bilstm_pred_48_lr.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.2465 - mae: 0.3720 - val_loss: 0.1347 - val_mae: 0.2813
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1279 - mae: 0.2756 - val_loss: 0.1082 - val_mae: 0.2497
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1099 - mae: 0.2545 - val_loss: 0.1091 - val_mae: 0.2570
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.1004 - mae: 0.2427 - val_loss: 0.0955 - val_mae: 0.2329
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0944 - mae: 0.2350 - val_loss: 0.0906 - val_mae: 0.2276
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0899 - mae: 0.2293 - val_loss: 0.0874 - val_mae: 0.2258
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0868 - mae: 0.2251 - val_loss: 0.0873 - val_mae: 0.2225
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - loss: 0.0839 - mae: 0.2211 - val_loss: 0.1000 - val_mae: 0.2468
Epoch 9/10
1586/1586 ━━━━━━━━━━━

{'MAE': 2227.3588036346764,
 'MSE': 8440569.798233192,
 'RMSE': np.float64(2905.2658739318836),
 'MAPE': 7.506272166462795,
 'R2': 0.7977731268135365,
 'Bias': np.float64(720.3134669698136)}

# Early stopping

In [115]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [116]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks = [early_stopping],
    epochs=20,
    batch_size=64
)

bilstm_pred_48_er = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_er.reshape(-1, 1)
).reshape(bilstm_pred_48_er.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - loss: 0.1538 - mae: 0.2958 - val_loss: 0.1017 - val_mae: 0.2413
Epoch 2/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0986 - mae: 0.2395 - val_loss: 0.0951 - val_mae: 0.2388
Epoch 3/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0885 - mae: 0.2266 - val_loss: 0.0994 - val_mae: 0.2450
Epoch 4/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0825 - mae: 0.2187 - val_loss: 0.0793 - val_mae: 0.2121
Epoch 5/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0787 - mae: 0.2132 - val_loss: 0.1117 - val_mae: 0.2604
Epoch 6/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0754 - mae: 0.2087 - val_loss: 0.0813 - val_mae: 0.2166
Epoch 7/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0727 - mae: 0.2048 - val_loss: 0.0782 - val_mae: 0.2115
Epoch 8/20
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - loss: 0.0703 - mae: 0.2017 - val_loss: 0.0801 - val_mae: 0.2123
Epoch 9/20
1586/1586 ━━━━━━━━━━━

{'MAE': 1829.4224290039122,
 'MSE': 6099368.564080156,
 'RMSE': np.float64(2469.6899732719808),
 'MAPE': 5.966216393770079,
 'R2': 0.853865762310984,
 'Bias': np.float64(84.7079144752107)}

# New Batch Size

In [117]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=32
)

bilstm_pred_48_bs = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bs.reshape(-1, 1)
).reshape(bilstm_pred_48_bs.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 27s 8ms/step - loss: 0.1390 - mae: 0.2809 - val_loss: 0.1098 - val_mae: 0.2586
Epoch 2/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0927 - mae: 0.2317 - val_loss: 0.0932 - val_mae: 0.2364
Epoch 3/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0839 - mae: 0.2200 - val_loss: 0.0830 - val_mae: 0.2150
Epoch 4/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0779 - mae: 0.2121 - val_loss: 0.0727 - val_mae: 0.2022
Epoch 5/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0741 - mae: 0.2067 - val_loss: 0.0791 - val_mae: 0.2153
Epoch 6/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0711 - mae: 0.2024 - val_loss: 0.0781 - val_mae: 0.2127
Epoch 7/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0680 - mae: 0.1981 - val_loss: 0.0790 - val_mae: 0.2113
Epoch 8/15
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 25s 8ms/step - loss: 0.0659 - mae: 0.1950 - val_loss: 0.0836 - val_mae: 0.2224
Epoch 9/15
3171/3171 ━━━━━━━━━━━

{'MAE': 1918.1593867896968,
 'MSE': 6381231.50726167,
 'RMSE': np.float64(2526.1099554971215),
 'MAPE': 6.426494760007206,
 'R2': 0.8471126327202941,
 'Bias': np.float64(787.9090135906599)}

In [118]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=16
)

bilstm_pred_48_bs = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_bs.reshape(-1, 1)
).reshape(bilstm_pred_48_bs.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 51s 8ms/step - loss: 0.1317 - mae: 0.2739 - val_loss: 0.1030 - val_mae: 0.2456
Epoch 2/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0904 - mae: 0.2285 - val_loss: 0.0778 - val_mae: 0.2114
Epoch 3/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 50s 8ms/step - loss: 0.0816 - mae: 0.2168 - val_loss: 0.0835 - val_mae: 0.2175
Epoch 4/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0761 - mae: 0.2092 - val_loss: 0.0822 - val_mae: 0.2132
Epoch 5/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0718 - mae: 0.2033 - val_loss: 0.0747 - val_mae: 0.2020
Epoch 6/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0686 - mae: 0.1987 - val_loss: 0.0880 - val_mae: 0.2204
Epoch 7/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0659 - mae: 0.1947 - val_loss: 0.0796 - val_mae: 0.2106
Epoch 8/15
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 49s 8ms/step - loss: 0.0639 - mae: 0.1915 - val_loss: 0.0822 - val_mae: 0.2157
Epoch 9/15
6341/6341 ━━━━━━━━━━━

{'MAE': 1885.0681512219612,
 'MSE': 6352819.467577709,
 'RMSE': np.float64(2520.480007375125),
 'MAPE': 6.267438034918585,
 'R2': 0.8477933542928282,
 'Bias': np.float64(536.4699647170785)}

# Adding Addtional Layers

In [121]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                64,
                return_sequences=True
            )
        ),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32)
        ),
    
        tf.keras.layers.BatchNormalization(),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Dense(64, activation="relu"),
    
        tf.keras.layers.Dense(24)
    ])

    return model

In [122]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

bilstm_pred_48_Al = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_Al.reshape(-1, 1)
).reshape(bilstm_pred_48_Al.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.2291 - mae: 0.3591 - val_loss: 0.1438 - val_mae: 0.2983
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1349 - mae: 0.2830 - val_loss: 0.1134 - val_mae: 0.2612
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1198 - mae: 0.2663 - val_loss: 0.0961 - val_mae: 0.2374
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1106 - mae: 0.2555 - val_loss: 0.0975 - val_mae: 0.2392
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.1050 - mae: 0.2488 - val_loss: 0.1093 - val_mae: 0.2508
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0992 - mae: 0.2420 - val_loss: 0.0841 - val_mae: 0.2172
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0949 - mae: 0.2367 - val_loss: 0.1197 - val_mae: 0.2601
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0912 - mae: 0.2323 - val_loss: 0.0895 - val_mae: 0.2295
Epoch 9/15
1586/1586 ━━━

{'MAE': 1975.2939845314254,
 'MSE': 7034777.14803121,
 'RMSE': np.float64(2652.3154314732647),
 'MAPE': 6.457290522288912,
 'R2': 0.8314543899029511,
 'Bias': np.float64(215.72153770373126)}

In [28]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                128,
                return_sequences=True
            )
        ),
    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

    
        tf.keras.layers.Dropout(0.5),
    
        tf.keras.layers.Dense(64, activation="relu"),
    
        tf.keras.layers.Dense(24)
    ])

    return model

In [36]:
bilstm_model_48 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_48.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_48.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

bilstm_pred_48_Al = bilstm_model_48.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_Al.reshape(-1, 1)
).reshape(bilstm_pred_48_Al.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.2079 - mae: 0.3346 - val_loss: 0.1620 - val_mae: 0.3248
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.1054 - mae: 0.2487 - val_loss: 0.0931 - val_mae: 0.2321
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0926 - mae: 0.2329 - val_loss: 0.0912 - val_mae: 0.2320
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0846 - mae: 0.2229 - val_loss: 0.0871 - val_mae: 0.2237
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - loss: 0.0786 - mae: 0.2147 - val_loss: 0.0881 - val_mae: 0.2236
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0731 - mae: 0.2076 - val_loss: 0.0839 - val_mae: 0.2169
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.0686 - mae: 0.2012 - val_loss: 0.1196 - val_mae: 0.2731
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 21s 13ms/step - loss: 0.0647 - mae: 0.1955 - val_loss: 0.0838 - val_mae: 0.2171
Epoch 9/15
1586/1586 ━━━

{'MAE': 1794.6315722437075,
 'MSE': 5976450.495023555,
 'RMSE': np.float64(2444.6779941381965),
 'MAPE': 5.893069101172644,
 'R2': 0.8568107455713129,
 'Bias': np.float64(15.376053424087653)}

In [37]:
bilstm_model_48.save("/kaggle/working/bilstm_model_48_RMS_85_15bias.keras")

In [23]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense
from tensorflow.keras.optimizers import Adam

In [41]:
def build_bilstm_hyper(hp):

    model = Sequential()

    model.add(
        Bidirectional(
            LSTM(
                units=hp.Choice(
                    "lstm_units",
                    values=[32, 64, 128]
                )
            ),
            input_shape=(
                X_train_seq.shape[1],
                X_train_seq.shape[2]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [43]:
tuner_bilstm = kt.RandomSearch(
    build_bilstm_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_bilstm.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64,
    verbose=1
)

Trial 5 Complete [00h 03m 27s]
val_loss: 0.08911152929067612

Best val_loss So Far: 0.07462777197360992
Total elapsed time: 00h 17m 31s


In [46]:

best_hp_bilstm = tuner_bilstm.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_bilstm

In [47]:
print("BEST Bi-LSTM HYPERPARAMETERS")

print(
    "LSTM Units:",
    best_hp_bilstm.get("lstm_units")
)

print(
    "Dense Units:",
    best_hp_bilstm.get("dense_units")
)

print(
    "Learning Rate:",
    best_hp_bilstm.get("learning_rate")
)

BEST Bi-LSTM HYPERPARAMETERS
LSTM Units: 64
Dense Units: 128
Learning Rate: 0.0005


In [48]:

best_bilstm_model = tuner_bilstm.get_best_models(
    num_models=1
)[0]

best_bilstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 128)            │        43,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,616 (244.59 KB)

 Trainable params: 62,616 (244.59 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
history_best_bilstm = best_bilstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=15,
    batch_size=64
)

Epoch 1/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.0387 - mae: 0.1443 - val_loss: 0.0796 - val_mae: 0.2064
Epoch 2/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0359 - mae: 0.1393 - val_loss: 0.0785 - val_mae: 0.2049
Epoch 3/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0338 - mae: 0.1354 - val_loss: 0.0757 - val_mae: 0.1995
Epoch 4/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0318 - mae: 0.1317 - val_loss: 0.0816 - val_mae: 0.2105
Epoch 5/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0300 - mae: 0.1283 - val_loss: 0.0834 - val_mae: 0.2098
Epoch 6/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0283 - mae: 0.1248 - val_loss: 0.0849 - val_mae: 0.2125
Epoch 7/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0270 - mae: 0.1222 - val_loss: 0.0836 - val_mae: 0.2089
Epoch 8/15
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0256 - mae: 0.1191 - val_loss: 0.0819 - val_mae: 0.2062
Epoch 9/15
1586/1586 ━━━━━━━━━━━

In [50]:

bilstm_pred_48_hy = best_bilstm_model.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_hy.reshape(-1, 1)
).reshape(bilstm_pred_48_hy.shape)

results = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

results

678/678 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 2068.2570119985744,
 'MSE': 7906878.712426445,
 'RMSE': np.float64(2811.917266284064),
 'MAPE': 6.818685039150524,
 'R2': 0.8105597848366451,
 'Bias': np.float64(584.4459662288199)}

In [ ]:
data = [
    ["Feature Engineered", "Base", 1983.9874887875158, 6996736.607804939, 2645.1345160133046, 6.674865726195193, 0.8323657998774167, 939.277266175341],

    ["Dropout", "0.2", 1942.9533460671366, 7083305.258758675, 2661.4479628124755, 6.408207644955419, 0.8302917091445837, 790.6286425787255],
    ["Dropout", "0.35", 1946.8032687056484, 6910368.552649346, 2628.757986701961, 6.444103102086086, 0.8344350845530754, 596.8464515947338],
    ["Dropout", "0.5", 1824.9897190871102, 6269557.12628878, 2503.9083701862533, 5.980970541785555, 0.8497882294417274, 360.0020746323614],

    ["Batch Normalization", "Added", 2190.5832167536705, 8063596.924461169, 2839.6473239578836, 7.38142629382956, 0.8068049869084544, 1121.0317592042652],

    ["Optimizer", "SGD", 2922.8521461639284, 13810281.711359901, 3716.218738362948, 9.818812926488706, 0.6691206689746094, 353.546182138574],
    ["Optimizer", "RMSprop", 1771.425878130128, 5619109.234146657, 2370.4660373324605, 5.793036501472333, 0.8653722535707811, 72.40864041789405],

    ["Learning Rate", "RMSprop 0.01", 2130.885912750419, 8005726.946277911, 2829.4393342635763, 6.992067660274932, 0.8081914886517158, 446.30317291764544],
    ["Learning Rate", "RMSprop 0.005", 2082.0503937970416, 7367050.406823056, 2714.231089429022, 7.015326774344978, 0.8234934839718115, 767.3909411308093],

    ["Early Stopping", "Enabled", 1829.4224290039122, 6099368.564080156, 2469.6899732719808, 5.966216393770079, 0.853865762310984, 84.7079144752107],

    ["Batch Size", "32", 1918.1593867896968, 6381231.50726167, 2526.1099554971215, 6.426494760007206, 0.8471126327202941, 787.9090135906599],
    ["Batch Size", "16", 1885.0681512219612, 6352819.467577709, 2520.480007375125, 6.267438034918585, 0.8477933542928282, 536.4699647170785],

    ["Additional Layers", "64, 32", 1975.2939845314254, 7034777.14803121, 2652.3154314732647, 6.457290522288912, 0.8314543899029511, 215.72153770373126],
    ["Additional Layers", "128, 64", 1794.6315722437075, 5976450.495023555, 2444.6779941381965, 5.893069101172644, 0.8568107455713129, 15.376053424087653],

    ["Hyperparameter Tuning", "Tuned", 2068.2570119985744, 7906878.712426445, 2811.917266284064, 6.818685039150524, 0.8105597848366451, 584.4459662288199],
]

df_bilstm_48 = pd.DataFrame(
    data,
    columns=[
        "Experiment",
        "Configuration",
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Bias"
    ]
)

df_bilstm_48

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
0,Feature Engineered,Base,1983.987489,6.996737e+06,2645.134516,6.674866,0.832366,939.277266
1,Dropout,0.2,1942.953346,7.083305e+06,2661.447963,6.408208,0.830292,790.628643
2,Dropout,0.35,1946.803269,6.910369e+06,2628.757987,6.444103,0.834435,596.846452
3,Dropout,0.5,1824.989719,6.269557e+06,2503.908370,5.980971,0.849788,360.002075
4,Batch Normalization,Added,2190.583217,8.063597e+06,2839.647324,7.381426,0.806805,1121.031759
5,Optimizer,SGD,2922.852146,1.381028e+07,3716.218738,9.818813,0.669121,353.546182
6,Optimizer,RMSprop,1771.425878,5.619109e+06,2370.466037,5.793037,0.865372,72.408640
7,Learning Rate,RMSprop 0.01,2130.885913,8.005727e+06,2829.439334,6.992068,0.808191,446.303173
8,Learning Rate,RMSprop 0.005,2082.050394,7.367050e+06,2714.231089,7.015327,0.823493,767.390941
9,Early Stopping,Enabled,1829.422429,6.099369e+06,2469.689973,5.966216,0.853866,84.707914


In [3]:
df_bilstm_48.sort_values(by = "R2", ascending=False)

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
6,Optimizer,RMSprop,1771.425878,5.619109e+06,2370.466037,5.793037,0.865372,72.408640
13,Additional Layers,"128, 64",1794.631572,5.976450e+06,2444.677994,5.893069,0.856811,15.376053
9,Early Stopping,Enabled,1829.422429,6.099369e+06,2469.689973,5.966216,0.853866,84.707914
3,Dropout,0.5,1824.989719,6.269557e+06,2503.908370,5.980971,0.849788,360.002075
11,Batch Size,16,1885.068151,6.352819e+06,2520.480007,6.267438,0.847793,536.469965
10,Batch Size,32,1918.159387,6.381232e+06,2526.109955,6.426495,0.847113,787.909014
2,Dropout,0.35,1946.803269,6.910369e+06,2628.757987,6.444103,0.834435,596.846452
0,Feature Engineered,Base,1983.987489,6.996737e+06,2645.134516,6.674866,0.832366,939.277266
12,Additional Layers,"64, 32",1975.293985,7.034777e+06,2652.315431,6.457291,0.831454,215.721538
1,Dropout,0.2,1942.953346,7.083305e+06,2661.447963,6.408208,0.830292,790.628643


# LSTM 168

# LAG Features

In [24]:
sequence_length = 168
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [25]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [28]:
lstm_model = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

I0000 00:00:1786445378.160444      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786445378.163563      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1074 - mae: 0.2361 - val_loss: 0.0913 - val_mae: 0.2298
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0586 - mae: 0.1795 - val_loss: 0.0921 - val_mae: 0.2295
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0510 - mae: 0.1663 - val_loss: 0.0790 - val_mae: 0.2081
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0462 - mae: 0.1581 - val_loss: 0.0885 - val_mae: 0.2223
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0423 - mae: 0.1512 - val_loss: 0.0795 - val_mae: 0.2071
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0393 - mae: 0.1459 - val_loss: 0.0826 - val_mae: 0.2126
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0366 - mae: 0.1411 - val_loss: 0.0852 - val_mae: 0.2145
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0342 - mae: 0.1367 - val_loss: 0.0851 - val_mae: 0.2123
Epoch 9/10
1584/1584 ━━━━━━━━━━

In [31]:
lstm_pred_168 = lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168.reshape(-1, 1)
).reshape(lstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2093.406195731432,
 'MSE': 8107396.7998581575,
 'RMSE': np.float64(2847.34908289415),
 'MAPE': 6.8988881241233635,
 'R2': 0.8062684460954157,
 'Bias': np.float64(792.6300895236636)}

# Drop Out

In [32]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [33]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1351 - mae: 0.2721 - val_loss: 0.1027 - val_mae: 0.2453
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0793 - mae: 0.2139 - val_loss: 0.0927 - val_mae: 0.2326
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0715 - mae: 0.2022 - val_loss: 0.0809 - val_mae: 0.2143
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0667 - mae: 0.1953 - val_loss: 0.0927 - val_mae: 0.2316
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0626 - mae: 0.1891 - val_loss: 0.0843 - val_mae: 0.2190
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0597 - mae: 0.1846 - val_loss: 0.0852 - val_mae: 0.2196
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0572 - mae: 0.1811 - val_loss: 0.0917 - val_mae: 0.2268
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0550 - mae: 0.1776 - val_loss: 0.0930 - val_mae: 0.2283
Epoch 9/10
1584/1584 ━━━━━━━━━

In [34]:
lstm_pred_168_dp = lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


{'MAE': 2093.406195731432,
 'MSE': 8107396.7998581575,
 'RMSE': np.float64(2847.34908289415),
 'MAPE': 6.8988881241233635,
 'R2': 0.8062684460954157,
 'Bias': np.float64(792.6300895236636)}

In [35]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [36]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1597 - mae: 0.2987 - val_loss: 0.1023 - val_mae: 0.2452
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0966 - mae: 0.2373 - val_loss: 0.0822 - val_mae: 0.2173
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0869 - mae: 0.2246 - val_loss: 0.0815 - val_mae: 0.2160
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0812 - mae: 0.2169 - val_loss: 0.0761 - val_mae: 0.2068
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0771 - mae: 0.2112 - val_loss: 0.0782 - val_mae: 0.2131
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0738 - mae: 0.2067 - val_loss: 0.0843 - val_mae: 0.2191
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0706 - mae: 0.2021 - val_loss: 0.0807 - val_mae: 0.2120
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0686 - mae: 0.1993 - val_loss: 0.0834 - val_mae: 0.2196
Epoch 9/10
1584/1584 ━━━━━━━━━

In [39]:
lstm_pred_168_dp = lstm_model_dp.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2102.219668501171,
 'MSE': 8191632.822401408,
 'RMSE': np.float64(2862.102867194226),
 'MAPE': 6.837262466515703,
 'R2': 0.8042555711930387,
 'Bias': np.float64(375.5403306296265)}

In [42]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [43]:
lstm_model_dp = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_dp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_dp.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1844 - mae: 0.3252 - val_loss: 0.1013 - val_mae: 0.2444
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1181 - mae: 0.2635 - val_loss: 0.1074 - val_mae: 0.2560
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1059 - mae: 0.2492 - val_loss: 0.0869 - val_mae: 0.2237
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1000 - mae: 0.2414 - val_loss: 0.0833 - val_mae: 0.2188
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0943 - mae: 0.2346 - val_loss: 0.0863 - val_mae: 0.2232
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0901 - mae: 0.2288 - val_loss: 0.0873 - val_mae: 0.2228
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0869 - mae: 0.2246 - val_loss: 0.0846 - val_mae: 0.2171
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0837 - mae: 0.2206 - val_loss: 0.0903 - val_mae: 0.2261
Epoch 9/10
1584/1584 ━━━━━━━━━━

In [44]:
lstm_pred_168_dp = lstm_model_dp.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_dp.reshape(-1, 1)
).reshape(lstm_pred_168_dp.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 1969.0540437155278,
 'MSE': 7186750.671312515,
 'RMSE': np.float64(2680.8115695275033),
 'MAPE': 6.300087888790882,
 'R2': 0.828267888022633,
 'Bias': np.float64(-291.70461006048976)}

# Adding Batch Normaliation

In [45]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [46]:
lstm_model_bn = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = lstm_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.2574 - mae: 0.3787 - val_loss: 0.1187 - val_mae: 0.2691
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1368 - mae: 0.2849 - val_loss: 0.0977 - val_mae: 0.2410
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1233 - mae: 0.2697 - val_loss: 0.0880 - val_mae: 0.2247
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1134 - mae: 0.2583 - val_loss: 0.0901 - val_mae: 0.2281
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1083 - mae: 0.2522 - val_loss: 0.0929 - val_mae: 0.2323
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.1034 - mae: 0.2462 - val_loss: 0.0900 - val_mae: 0.2278
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0993 - mae: 0.2413 - val_loss: 0.0880 - val_mae: 0.2261
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 0.0961 - mae: 0.2373 - val_loss: 0.0888 - val_mae: 0.2275
Epoch 9/10
1584/1584 ━━━

In [47]:
lstm_pred_168_bn = lstm_model_bn.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_bn.reshape(-1, 1)
).reshape(lstm_pred_168_bn.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2070.2316370186686,
 'MSE': 7643918.283221489,
 'RMSE': np.float64(2764.7636939206013),
 'MAPE': 6.864982824811837,
 'R2': 0.8173435686589227,
 'Bias': np.float64(790.5956146045879)}

# New OPtimizers

In [62]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(64),


        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [53]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.SGD(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)


In [55]:

history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - loss: 1.2169 - mae: 0.8538 - val_loss: 0.4756 - val_mae: 0.5451
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.7415 - mae: 0.6739 - val_loss: 0.3599 - val_mae: 0.4740
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.6045 - mae: 0.6089 - val_loss: 0.3104 - val_mae: 0.4407
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.5264 - mae: 0.5691 - val_loss: 0.2821 - val_mae: 0.4202
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4748 - mae: 0.5406 - val_loss: 0.2648 - val_mae: 0.4071
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4379 - mae: 0.5193 - val_loss: 0.2515 - val_mae: 0.3965
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.4094 - mae: 0.5021 - val_loss: 0.2421 - val_mae: 0.3886
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.3877 - mae: 0.4885 - val_loss: 0.2348 - val_mae: 0.3826
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2817.8998274735054,
 'MSE': 13025983.907269912,
 'RMSE': np.float64(3609.1527963318363),
 'MAPE': 9.57642785472432,
 'R2': 0.6887355872928707,
 'Bias': np.float64(604.3252836178002)}

In [56]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.2350 - mae: 0.3642 - val_loss: 0.1197 - val_mae: 0.2652
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1374 - mae: 0.2853 - val_loss: 0.1210 - val_mae: 0.2735
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1240 - mae: 0.2703 - val_loss: 0.0958 - val_mae: 0.2347
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1158 - mae: 0.2611 - val_loss: 0.1040 - val_mae: 0.2512
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1111 - mae: 0.2556 - val_loss: 0.1117 - val_mae: 0.2457
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1058 - mae: 0.2492 - val_loss: 0.1021 - val_mae: 0.2420
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1020 - mae: 0.2446 - val_loss: 0.0958 - val_mae: 0.2394
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0987 - mae: 0.2407 - val_loss: 0.1064 - val_mae: 0.2505
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2025.5845268342443,
 'MSE': 7116907.695226848,
 'RMSE': np.float64(2667.7533047916645),
 'MAPE': 6.561243845127129,
 'R2': 0.8299368316577383,
 'Bias': np.float64(14.304081307266548)}

# Changing Learning Rate 

In [57]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1969 - mae: 0.3351 - val_loss: 0.2340 - val_mae: 0.3696
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1374 - mae: 0.2837 - val_loss: 0.1251 - val_mae: 0.2727
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1269 - mae: 0.2725 - val_loss: 0.1141 - val_mae: 0.2563
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1213 - mae: 0.2664 - val_loss: 0.1265 - val_mae: 0.2760
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1173 - mae: 0.2622 - val_loss: 0.1221 - val_mae: 0.2615
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1153 - mae: 0.2597 - val_loss: 0.1326 - val_mae: 0.2804
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1138 - mae: 0.2578 - val_loss: 0.1197 - val_mae: 0.2664
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1124 - mae: 0.2564 - val_loss: 0.1147 - val_mae: 0.2592
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2286.1492992531557,
 'MSE': 8918676.523530735,
 'RMSE': np.float64(2986.4153300454936),
 'MAPE': 7.491368469178184,
 'R2': 0.7868823860075315,
 'Bias': np.float64(377.8933612000542)}

In [58]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.2965 - mae: 0.4082 - val_loss: 0.1296 - val_mae: 0.2775
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1511 - mae: 0.3003 - val_loss: 0.1080 - val_mae: 0.2524
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1342 - mae: 0.2823 - val_loss: 0.0971 - val_mae: 0.2378
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1251 - mae: 0.2718 - val_loss: 0.1003 - val_mae: 0.2443
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1190 - mae: 0.2649 - val_loss: 0.0913 - val_mae: 0.2313
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1145 - mae: 0.2596 - val_loss: 0.0910 - val_mae: 0.2302
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1106 - mae: 0.2550 - val_loss: 0.1059 - val_mae: 0.2474
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1078 - mae: 0.2517 - val_loss: 0.0935 - val_mae: 0.2324
Epoch 9/10
1584/1584 ━━━━━━━━━━

{'MAE': 2055.2365116455967,
 'MSE': 7291163.783825497,
 'RMSE': np.float64(2700.2155069226415),
 'MAPE': 6.894155457428436,
 'R2': 0.8257728683468342,
 'Bias': np.float64(756.8230088590758)}

# Early Stoping

In [59]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [60]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.01),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks = [early_stopping],
    epochs=10,
    batch_size=64
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.1944 - mae: 0.3331 - val_loss: 0.1567 - val_mae: 0.3085
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1376 - mae: 0.2838 - val_loss: 0.1289 - val_mae: 0.2804
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1280 - mae: 0.2736 - val_loss: 0.1226 - val_mae: 0.2735
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1226 - mae: 0.2682 - val_loss: 0.1325 - val_mae: 0.2851
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1180 - mae: 0.2630 - val_loss: 0.1703 - val_mae: 0.3264
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1150 - mae: 0.2598 - val_loss: 0.1406 - val_mae: 0.2831
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1115 - mae: 0.2556 - val_loss: 0.2089 - val_mae: 0.3685
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.1114 - mae: 0.2552 - val_loss: 0.1482 - val_mae: 0.2920
674/674 ━━━━━━━━━━━━━━━━━━━━ 2s

{'MAE': 2400.126212042043,
 'MSE': 9519815.741643934,
 'RMSE': np.float64(3085.4198647256962),
 'MAPE': 8.027378691509938,
 'R2': 0.7725177708649626,
 'Bias': np.float64(789.793445600131)}

# New Batch Size

In [61]:
history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=32
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.1476 - mae: 0.2947 - val_loss: 0.1442 - val_mae: 0.2992
Epoch 2/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1450 - mae: 0.2921 - val_loss: 0.1394 - val_mae: 0.2919
Epoch 3/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1507 - mae: 0.2974 - val_loss: 0.1387 - val_mae: 0.2871
Epoch 4/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1491 - mae: 0.2961 - val_loss: 0.1424 - val_mae: 0.2931
Epoch 5/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.1519 - mae: 0.2983 - val_loss: 0.1313 - val_mae: 0.2829
Epoch 6/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.2114 - mae: 0.3553 - val_loss: 0.2217 - val_mae: 0.3668
Epoch 7/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - loss: 0.3149 - mae: 0.4383 - val_loss: 0.2608 - val_mae: 0.3981
Epoch 8/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 30s 9ms/step - loss: 0.3644 - mae: 0.4734 - val_loss: 0.3150 - val_mae: 0.4431
Epoch 9/10
3167/3167 ━━━━━━━━━━━

{'MAE': 3252.461922991529,
 'MSE': 17352189.20721325,
 'RMSE': np.float64(4165.59590061413),
 'MAPE': 10.626736239941604,
 'R2': 0.5853580795726447,
 'Bias': np.float64(-342.31112114060664)}

In [63]:
history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=16
)

lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

results


Epoch 1/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5163 - mae: 0.5604 - val_loss: 0.4079 - val_mae: 0.5084
Epoch 2/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5285 - mae: 0.5692 - val_loss: 0.4212 - val_mae: 0.5136
Epoch 3/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5445 - mae: 0.5783 - val_loss: 0.4169 - val_mae: 0.5153
Epoch 4/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5424 - mae: 0.5788 - val_loss: 0.4409 - val_mae: 0.5316
Epoch 5/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5343 - mae: 0.5748 - val_loss: 0.4073 - val_mae: 0.5056
Epoch 6/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 57s 9ms/step - loss: 0.5370 - mae: 0.5762 - val_loss: 0.4121 - val_mae: 0.5071
Epoch 7/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5438 - mae: 0.5784 - val_loss: 0.4231 - val_mae: 0.5123
Epoch 8/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 58s 9ms/step - loss: 0.5370 - mae: 0.5738 - val_loss: 0.4247 - val_mae: 0.5173
Epoch 9/10
6334/6334 ━━━━━━━━━━━

{'MAE': 3382.526341136323,
 'MSE': 18625296.021118943,
 'RMSE': np.float64(4315.703421357744),
 'MAPE': 11.257215633140486,
 'R2': 0.5549363588362453,
 'Bias': np.float64(155.9917540118888)}

# Adding Additional Layers

In [69]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(
            64,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.LSTM(
            32
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(32, activation="relu"),

        tf.keras.layers.Dense(1)
    ])

    return model

In [70]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.5911 - mae: 0.6157 - val_loss: 0.4709 - val_mae: 0.5407
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5661 - mae: 0.6049 - val_loss: 0.4745 - val_mae: 0.5425
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5587 - mae: 0.6018 - val_loss: 0.4880 - val_mae: 0.5450
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5543 - mae: 0.6001 - val_loss: 0.4847 - val_mae: 0.5406
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5513 - mae: 0.5989 - val_loss: 0.4826 - val_mae: 0.5410
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5489 - mae: 0.5978 - val_loss: 0.5067 - val_mae: 0.5539
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5467 - mae: 0.5968 - val_loss: 0.5121 - val_mae: 0.5573
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 22s 14ms/step - loss: 0.5449 - mae: 0.5960 - val_loss: 0.5270 - val_mae: 0.5663
Epoch 9/10
1584/1584 ━━━

ValueError: Found input variables with inconsistent numbers of samples: [517536, 21564]

In [77]:
print("y_test_original shape:", y_test_original.shape)
print("lstm_pred_168_no shape:", lstm_pred_168_no.shape)
print("lstm_pred_original_168 shape:", lstm_pred_original_168.shape)

print("y_test_original size:", np.asarray(y_test_original).size)
print("prediction size:", np.asarray(lstm_pred_original_168).size)

y_test_original shape: (21564, 24, 1)
lstm_pred_168_no shape: (21564, 1)
lstm_pred_original_168 shape: (21564, 1)
y_test_original size: 517536
prediction size: 21564


In [21]:
y_test_original_168 = y_test_original[:, -1, :]

print(y_test_original_168.shape)

(21684, 1)


In [80]:
lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 4056.809134835548,
 'MSE': 27195216.37617009,
 'RMSE': np.float64(5214.903294996954),
 'MAPE': 13.924634126834937,
 'R2': 0.3507844731481431,
 'Bias': np.float64(1303.0323106146238)}

In [27]:
def build_lstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.LSTM(
            128,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.LSTM(
            64
        ),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(32, activation="relu"),

        tf.keras.layers.Dense(1)
    ])

    return model

In [28]:
lstm_model_no = build_lstm_model(
    sequence_length=sequence_length,
    n_features=19
)

lstm_model_no.compile(
    optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.0005),
    loss="mse",
    metrics=["mae"]
)


history = lstm_model_no.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


E0000 00:00:1786531845.478729  593804 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1786531850.597790  593804 cpu_allocator_impl.cc:82] Allocation of 1293781440 exceeds 10% of free system memory.


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 0s 403ms/step - loss: 0.5810 - mae: 0.6117

W0000 00:00:1786532495.698178  593804 cpu_allocator_impl.cc:82] Allocation of 275316384 exceeds 10% of free system memory.


1584/1584 ━━━━━━━━━━━━━━━━━━━━ 726s 456ms/step - loss: 0.5810 - mae: 0.6117 - val_loss: 0.4758 - val_mae: 0.5357
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 894s 564ms/step - loss: 0.5607 - mae: 0.6027 - val_loss: 0.4709 - val_mae: 0.5378
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 777s 491ms/step - loss: 0.5550 - mae: 0.6004 - val_loss: 0.4730 - val_mae: 0.5353
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 699s 442ms/step - loss: 0.5511 - mae: 0.5985 - val_loss: 0.4705 - val_mae: 0.5366
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 673s 425ms/step - loss: 0.5479 - mae: 0.5971 - val_loss: 0.4668 - val_mae: 0.5335
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 942s 595ms/step - loss: 0.5450 - mae: 0.5959 - val_loss: 0.4777 - val_mae: 0.5352
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 841s 505ms/step - loss: 0.5427 - mae: 0.5948 - val_loss: 0.4840 - val_mae: 0.5375
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 515s 325ms/step - loss: 0.5403 - mae: 0.5938 - val_loss: 0.4704 - val_mae: 0.5356
Epoch 9/10
1584/158

In [86]:
lstm_pred_168_no = lstm_model_no.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_no.reshape(-1, 1)
).reshape(lstm_pred_168_no.shape)

results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)

results




674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 4099.808879780827,
 'MSE': 27722606.824187826,
 'RMSE': np.float64(5265.226189271248),
 'MAPE': 14.176232914515655,
 'R2': 0.3381943888174852,
 'Bias': np.float64(1595.3570324093396)}

In [27]:
def build_lstm_hyper(hp):

    model = Sequential()

    model.add(
        LSTM(
            units=hp.Choice(
                "lstm_units",
                values=[32, 64, 128]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [28]:
tuner_lstm = kt.RandomSearch(
    build_lstm_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_lstm.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64,
    verbose=1
)

Trial 5 Complete [00h 02m 37s]
val_loss: 0.07770843803882599

Best val_loss So Far: 0.0770488828420639
Total elapsed time: 00h 12m 32s


In [29]:
best_hp_lstm = tuner_lstm.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_lstm

In [31]:
best_lstm_model = tuner_lstm.get_best_models(
    num_models=1
)[0]

best_lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        75,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,592 (334.34 KB)

 Trainable params: 85,592 (334.34 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
history_best_lstm = best_lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0328 - mae: 0.1346 - val_loss: 0.0990 - val_mae: 0.2390
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0292 - mae: 0.1276 - val_loss: 0.0837 - val_mae: 0.2121
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0262 - mae: 0.1214 - val_loss: 0.0838 - val_mae: 0.2132
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0239 - mae: 0.1162 - val_loss: 0.0881 - val_mae: 0.2176
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0218 - mae: 0.1113 - val_loss: 0.0888 - val_mae: 0.2199
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0202 - mae: 0.1074 - val_loss: 0.0812 - val_mae: 0.2050
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - loss: 0.0187 - mae: 0.1035 - val_loss: 0.0861 - val_mae: 0.2143
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 15s 10ms/step - loss: 0.0175 - mae: 0.1003 - val_loss: 0.0814 - val_mae: 0.2049
Epoch 9/10
1584/1584 ━━━━━━━

In [31]:
y_test_original_168 = y_scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).reshape(y_test_seq.shape)

y_test_original_168 = y_test_original_168.squeeze(-1)

In [51]:
lstm_pred_168_hy = best_lstm_model.predict(
    X_test_seq
)

lstm_pred_original_168 = y_scaler.inverse_transform(
    lstm_pred_168_hy.reshape(-1, 1)
).reshape(lstm_pred_168_hy.shape)


674/674 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [52]:
results = evaluate_deep_model(
    y_test_original_168,
    lstm_pred_original_168
)
results

{'MAE': 1947.5263563790731,
 'MSE': 6851497.791196226,
 'RMSE': np.float64(2617.5365883204436),
 'MAPE': 6.4726604688660565,
 'R2': 0.8362789750607131,
 'Bias': np.float64(696.291991417626)}

In [4]:
data = [
    ["Feature Engineered", "Base", 2093.406195731432, 8107396.7998581575, 2847.34908289415, 6.8988881241233635, 0.8062684460954157, 792.6300895236636],

    ["Dropout", "0.2", 2093.406195731432, 8107396.7998581575, 2847.34908289415, 6.8988881241233635, 0.8062684460954157],
    ["Dropout", "0.35", 2102.219668501171, 8191632.822401408, 2862.102867194226, 6.837262466515703, 0.8042555711930387, 375.5403306296265],
    ["Dropout", "0.5", 1969.0540437155278, 7186750.671312515, 2680.8115695275033, 6.300087888790882, 0.828267888022633, -291.70461006048976],

    ["Batch Normalization", "Added", 2070.2316370186686, 7643918.283221489, 2764.7636939206013, 6.864982824811837, 0.8173435686589227, 790.5956146045879],

    ["Optimizer", "SGD", 2817.8998274735054, 13025983.907269912, 3609.1527963318363, 9.57642785472432, 0.6887355872928707, 604.3252836178002],
    ["Optimizer", "RMSprop", 2025.5845268342443, 7116907.695226848, 2667.7533047916645, 6.561243845127129, 0.8299368316577383, 14.304081307266548],

    ["Learning Rate", "RMSprop 0.01", 2286.1492992531557, 8918676.523530735, 2986.4153300454936, 7.491368469178184, 0.7868823860075315, 377.8933612000542],
    ["Learning Rate", "RMSprop 0.005", 2055.2365116455967, 7291163.783825497, 2700.2155069226415, 6.894155457428436, 0.8257728683468342, 756.8230088590758],

    ["Early Stopping", "Enabled", 2400.126212042043, 9519815.741643934, 3085.4198647256962, 8.027378691509938, 0.7725177708649626, 789.793445600131],

    ["Batch Size", "32", 3252.461922991529, 17352189.20721325, 4165.59590061413, 10.626736239941604, 0.5853580795726447, -342.31112114060664],
    ["Batch Size", "16", 3382.526341136323, 18625296.021118943, 4315.703421357744, 11.257215633140486, 0.5549363588362453, 155.9917540118888],

    ["Additional Layers", "64, 32", 4056.809134835548, 27195216.37617009, 5214.903294996954, 13.924634126834937, 0.3507844731481431, 1303.0323106146238],
    ["Additional Layers", "128, 64", 4099.808879780827, 27722606.824187826, 5265.226189271248, 14.176232914515655, 0.3381943888174852, 1595.3570324093396],

    ["Hyperparameter Tuning", "Tuned", 1947.5263563790731, 6851497.791196226, 2617.5365883204436, 6.4726604688660565, 0.8362789750607131, 696.291991417626],
]

df_lstm_168 = pd.DataFrame(
    data,
    columns=[
        "Experiment",
        "Configuration",
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Bias"
    ]
)

df_lstm_168

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
0,Feature Engineered,Base,2093.406196,8.107397e+06,2847.349083,6.898888,0.806268,792.630090
1,Dropout,0.2,2093.406196,8.107397e+06,2847.349083,6.898888,0.806268,NaN
2,Dropout,0.35,2102.219669,8.191633e+06,2862.102867,6.837262,0.804256,375.540331
3,Dropout,0.5,1969.054044,7.186751e+06,2680.811570,6.300088,0.828268,-291.704610
4,Batch Normalization,Added,2070.231637,7.643918e+06,2764.763694,6.864983,0.817344,790.595615
5,Optimizer,SGD,2817.899827,1.302598e+07,3609.152796,9.576428,0.688736,604.325284
6,Optimizer,RMSprop,2025.584527,7.116908e+06,2667.753305,6.561244,0.829937,14.304081
7,Learning Rate,RMSprop 0.01,2286.149299,8.918677e+06,2986.415330,7.491368,0.786882,377.893361
8,Learning Rate,RMSprop 0.005,2055.236512,7.291164e+06,2700.215507,6.894155,0.825773,756.823009
9,Early Stopping,Enabled,2400.126212,9.519816e+06,3085.419865,8.027379,0.772518,789.793446


In [5]:
df_lstm_168.sort_values(by = "R2", ascending=False)

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
14,Hyperparameter Tuning,Tuned,1947.526356,6.851498e+06,2617.536588,6.472660,0.836279,696.291991
6,Optimizer,RMSprop,2025.584527,7.116908e+06,2667.753305,6.561244,0.829937,14.304081
3,Dropout,0.5,1969.054044,7.186751e+06,2680.811570,6.300088,0.828268,-291.704610
8,Learning Rate,RMSprop 0.005,2055.236512,7.291164e+06,2700.215507,6.894155,0.825773,756.823009
4,Batch Normalization,Added,2070.231637,7.643918e+06,2764.763694,6.864983,0.817344,790.595615
0,Feature Engineered,Base,2093.406196,8.107397e+06,2847.349083,6.898888,0.806268,792.630090
1,Dropout,0.2,2093.406196,8.107397e+06,2847.349083,6.898888,0.806268,NaN
2,Dropout,0.35,2102.219669,8.191633e+06,2862.102867,6.837262,0.804256,375.540331
7,Learning Rate,RMSprop 0.01,2286.149299,8.918677e+06,2986.415330,7.491368,0.786882,377.893361
9,Early Stopping,Enabled,2400.126212,9.519816e+06,3085.419865,8.027379,0.772518,789.793446


# 168 RNN

In [26]:
sequence_length = 168
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [27]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [28]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



I0000 00:00:1786552269.420336      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786552269.423392      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
  16/1584 ━━━━━━━━━━━━━━━━━━━━ 17s 11ms/step - loss: 1.0575 - mae: 0.8060

I0000 00:00:1786552276.830134     157 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.1285 - mae: 0.2612 - val_loss: 0.1225 - val_mae: 0.2608
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0757 - mae: 0.2049 - val_loss: 0.1083 - val_mae: 0.2460
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0659 - mae: 0.1895 - val_loss: 0.1114 - val_mae: 0.2492
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0617 - mae: 0.1825 - val_loss: 0.0983 - val_mae: 0.2351
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0585 - mae: 0.1771 - val_loss: 0.1032 - val_mae: 0.2371
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0558 - mae: 0.1729 - val_loss: 0.0947 - val_mae: 0.2274
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0535 - mae: 0.1693 - val_loss: 0.0965 - val_mae: 0.2304
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0515 - mae: 0.1661 - val_loss: 0.1090 - val_mae: 0.2435
Epoch 9/10
1584/1584 ━━━━━━━━━━━━━━

In [31]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2480.208866380491,
 'MSE': 10927148.972322095,
 'RMSE': np.float64(3305.6238401128003),
 'MAPE': 8.410324175007107,
 'R2': 0.7388886220307018,
 'Bias': np.float64(1228.078319740378)}

# Drop out

In [32]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.2),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [33]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.1633 - mae: 0.3010 - val_loss: 0.1128 - val_mae: 0.2508
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1017 - mae: 0.2426 - val_loss: 0.1137 - val_mae: 0.2557
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0907 - mae: 0.2281 - val_loss: 0.0932 - val_mae: 0.2266
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0846 - mae: 0.2200 - val_loss: 0.0937 - val_mae: 0.2253
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0802 - mae: 0.2139 - val_loss: 0.0896 - val_mae: 0.2213
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0777 - mae: 0.2102 - val_loss: 0.0847 - val_mae: 0.2141
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0753 - mae: 0.2071 - val_loss: 0.0875 - val_mae: 0.2186
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0732 - mae: 0.2040 - val_loss: 0.0849 - val_mae: 0.2115
Epoch 9/10
1584/1584 ━━━

In [35]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 1797.5445667555312,
 'MSE': 5850402.645245208,
 'RMSE': np.float64(2418.7605597175607),
 'MAPE': 5.949787595291549,
 'R2': 0.8602007989234381,
 'Bias': np.float64(565.7434255215156)}

In [36]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.35),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [37]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.1916 - mae: 0.3269 - val_loss: 0.1238 - val_mae: 0.2644
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1238 - mae: 0.2688 - val_loss: 0.1084 - val_mae: 0.2477
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1107 - mae: 0.2540 - val_loss: 0.1104 - val_mae: 0.2515
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1038 - mae: 0.2456 - val_loss: 0.0955 - val_mae: 0.2281
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0986 - mae: 0.2391 - val_loss: 0.1029 - val_mae: 0.2403
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0948 - mae: 0.2339 - val_loss: 0.0969 - val_mae: 0.2301
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0912 - mae: 0.2295 - val_loss: 0.0965 - val_mae: 0.2312
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.0891 - mae: 0.2266 - val_loss: 0.1051 - val_mae: 0.2361
Epoch 9/10
1584/1584 ━━━

In [38]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2255.6494169834923,
 'MSE': 8691934.225433037,
 'RMSE': np.float64(2948.2086468621987),
 'MAPE': 7.450667563906278,
 'R2': 0.7923005416536363,
 'Bias': np.float64(64.64306670061357)}

In [39]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [40]:
rnn_model = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.2229 - mae: 0.3570 - val_loss: 0.1260 - val_mae: 0.2689
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1478 - mae: 0.2952 - val_loss: 0.1173 - val_mae: 0.2605
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1355 - mae: 0.2827 - val_loss: 0.0998 - val_mae: 0.2381
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1265 - mae: 0.2725 - val_loss: 0.1057 - val_mae: 0.2441
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1198 - mae: 0.2647 - val_loss: 0.1006 - val_mae: 0.2392
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1151 - mae: 0.2595 - val_loss: 0.1126 - val_mae: 0.2546
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1113 - mae: 0.2548 - val_loss: 0.0987 - val_mae: 0.2376
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1083 - mae: 0.2513 - val_loss: 0.0999 - val_mae: 0.2361
Epoch 9/10
1584/1584 ━━━

In [43]:
rnn_model_168 = rnn_model.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step


{'MAE': 1805.7853438086613,
 'MSE': 5851812.077466108,
 'RMSE': np.float64(2419.051896397865),
 'MAPE': 5.883813948274826,
 'R2': 0.8601671196178584,
 'Bias': np.float64(-31.659633388577074)}

# Batch Normaliztion

In [44]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [45]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.2829 - mae: 0.3975 - val_loss: 0.1413 - val_mae: 0.2833
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1623 - mae: 0.3104 - val_loss: 0.1183 - val_mae: 0.2591
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1470 - mae: 0.2949 - val_loss: 0.1234 - val_mae: 0.2672
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1361 - mae: 0.2834 - val_loss: 0.1117 - val_mae: 0.2575
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1266 - mae: 0.2728 - val_loss: 0.1375 - val_mae: 0.2918
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1203 - mae: 0.2662 - val_loss: 0.0992 - val_mae: 0.2376
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1163 - mae: 0.2616 - val_loss: 0.1072 - val_mae: 0.2486
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1129 - mae: 0.2576 - val_loss: 0.1283 - val_mae: 0.2760
Epoch 9/10
1584/1584 ━━━

In [46]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2322.207466017461,
 'MSE': 9269023.23703972,
 'RMSE': np.float64(3044.507059778269),
 'MAPE': 7.3228278024852465,
 'R2': 0.7785106219396067,
 'Bias': np.float64(-878.5953776645489)}

# New Optimizers

In [47]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.2580 - mae: 0.3810 - val_loss: 0.1561 - val_mae: 0.2980
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1645 - mae: 0.3124 - val_loss: 0.1287 - val_mae: 0.2699
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1523 - mae: 0.3000 - val_loss: 0.1192 - val_mae: 0.2617
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1432 - mae: 0.2908 - val_loss: 0.1259 - val_mae: 0.2747
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1359 - mae: 0.2828 - val_loss: 0.1306 - val_mae: 0.2779
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1292 - mae: 0.2759 - val_loss: 0.1489 - val_mae: 0.2923
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1231 - mae: 0.2689 - val_loss: 0.1661 - val_mae: 0.3156
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1194 - mae: 0.2648 - val_loss: 0.1667 - val_mae: 0.3187
Epoch 9/10
1584/1584 ━━━

In [48]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2458.7868039237246,
 'MSE': 11013649.120632723,
 'RMSE': np.float64(3318.6818348001852),
 'MAPE': 7.559939461626539,
 'R2': 0.7368216443609413,
 'Bias': np.float64(-1251.5776093202937)}

In [49]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer="sgd",
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.6013 - mae: 0.5934 - val_loss: 0.2214 - val_mae: 0.3678
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.3118 - mae: 0.4352 - val_loss: 0.1811 - val_mae: 0.3289
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2627 - mae: 0.3973 - val_loss: 0.1661 - val_mae: 0.3128
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2375 - mae: 0.3772 - val_loss: 0.1583 - val_mae: 0.3052
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2219 - mae: 0.3640 - val_loss: 0.1503 - val_mae: 0.2966
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2097 - mae: 0.3535 - val_loss: 0.1458 - val_mae: 0.2906
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2008 - mae: 0.3461 - val_loss: 0.1408 - val_mae: 0.2854
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1916 - mae: 0.3378 - val_loss: 0.1388 - val_mae: 0.2842
Epoch 9/10
1584/1584 ━━━

In [50]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2233.103843572446,
 'MSE': 8596216.821249258,
 'RMSE': np.float64(2931.9305621465965),
 'MAPE': 7.556091483349981,
 'R2': 0.7945877716864085,
 'Bias': np.float64(747.6869103180443)}

# Changing Learning Rate

In [51]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.3722 - mae: 0.4544 - val_loss: 0.1584 - val_mae: 0.3072
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 12ms/step - loss: 0.1818 - mae: 0.3293 - val_loss: 0.1319 - val_mae: 0.2804
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1574 - mae: 0.3062 - val_loss: 0.1212 - val_mae: 0.2696
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1448 - mae: 0.2930 - val_loss: 0.1023 - val_mae: 0.2462
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1373 - mae: 0.2847 - val_loss: 0.1281 - val_mae: 0.2826
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1316 - mae: 0.2788 - val_loss: 0.0980 - val_mae: 0.2366
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1287 - mae: 0.2754 - val_loss: 0.0973 - val_mae: 0.2393
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1254 - mae: 0.2717 - val_loss: 0.0948 - val_mae: 0.2330
Epoch 9/10
1584/1584 ━━━

In [52]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 1757.302142847611,
 'MSE': 5535204.282548363,
 'RMSE': np.float64(2352.70148606838),
 'MAPE': 5.722750542844209,
 'R2': 0.8677326701394256,
 'Bias': np.float64(-109.67954021254077)}

In [53]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.00025),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.4903 - mae: 0.5265 - val_loss: 0.1742 - val_mae: 0.3238
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2203 - mae: 0.3630 - val_loss: 0.1457 - val_mae: 0.2941
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1799 - mae: 0.3280 - val_loss: 0.1322 - val_mae: 0.2791
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1638 - mae: 0.3123 - val_loss: 0.1325 - val_mae: 0.2788
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1554 - mae: 0.3040 - val_loss: 0.1420 - val_mae: 0.2902
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1486 - mae: 0.2974 - val_loss: 0.1270 - val_mae: 0.2752
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1433 - mae: 0.2914 - val_loss: 0.1081 - val_mae: 0.2525
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1384 - mae: 0.2865 - val_loss: 0.1193 - val_mae: 0.2661
Epoch 9/10
1584/1584 ━━━

In [54]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2169.64029475048,
 'MSE': 8161735.031777124,
 'RMSE': np.float64(2856.875046580988),
 'MAPE': 7.250662821874354,
 'R2': 0.8049699984720955,
 'Bias': np.float64(310.486800672894)}

# Early Stopping

In [55]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [56]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.00025),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks=early_stopping,
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.4885 - mae: 0.5242 - val_loss: 0.1682 - val_mae: 0.3158
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.2157 - mae: 0.3598 - val_loss: 0.1380 - val_mae: 0.2846
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1786 - mae: 0.3264 - val_loss: 0.1302 - val_mae: 0.2769
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1632 - mae: 0.3118 - val_loss: 0.1213 - val_mae: 0.2669
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1548 - mae: 0.3033 - val_loss: 0.1212 - val_mae: 0.2614
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1464 - mae: 0.2948 - val_loss: 0.1173 - val_mae: 0.2568
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1421 - mae: 0.2898 - val_loss: 0.1317 - val_mae: 0.2726
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - loss: 0.1383 - mae: 0.2857 - val_loss: 0.1100 - val_mae: 0.2512
Epoch 9/10
1584/1584 ━━━

In [57]:
rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2461.1576237533172,
 'MSE': 11584741.290556297,
 'RMSE': np.float64(3403.636480377465),
 'MAPE': 8.039282919860984,
 'R2': 0.7231750231046621,
 'Bias': np.float64(-235.05391399176216)}

# New Batch Size

In [59]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks=early_stopping,
    epochs=10,
    batch_size=32
)

rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


Epoch 1/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 44s 13ms/step - loss: 0.3243 - mae: 0.4260 - val_loss: 0.1427 - val_mae: 0.2926
Epoch 2/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.1824 - mae: 0.3299 - val_loss: 0.1187 - val_mae: 0.2623
Epoch 3/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.1634 - mae: 0.3121 - val_loss: 0.1246 - val_mae: 0.2726
Epoch 4/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.1521 - mae: 0.3002 - val_loss: 0.1155 - val_mae: 0.2608
Epoch 5/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 38s 12ms/step - loss: 0.1451 - mae: 0.2925 - val_loss: 0.1213 - val_mae: 0.2656
674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 2328.7370715439965,
 'MSE': 9147753.439265924,
 'RMSE': np.float64(3024.5253246197035),
 'MAPE': 7.818215398208486,
 'R2': 0.7814084431446604,
 'Bias': np.float64(378.7625200680484)}

In [61]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),    
    epochs=10,
    batch_size=16
)

rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


Epoch 1/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 77s 12ms/step - loss: 0.2992 - mae: 0.4148 - val_loss: 0.1300 - val_mae: 0.2754
Epoch 2/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 70s 11ms/step - loss: 0.1938 - mae: 0.3408 - val_loss: 0.1303 - val_mae: 0.2782
Epoch 3/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1733 - mae: 0.3216 - val_loss: 0.1310 - val_mae: 0.2786
Epoch 4/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1580 - mae: 0.3059 - val_loss: 0.1081 - val_mae: 0.2514
Epoch 5/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1470 - mae: 0.2949 - val_loss: 0.1129 - val_mae: 0.2570
Epoch 6/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1392 - mae: 0.2868 - val_loss: 0.1206 - val_mae: 0.2656
Epoch 7/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1360 - mae: 0.2833 - val_loss: 0.1121 - val_mae: 0.2552
Epoch 8/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 69s 11ms/step - loss: 0.1318 - mae: 0.2792 - val_loss: 0.1086 - val_mae: 0.2509
Epoch 9/10
6334/6334 ━━━

{'MAE': 1925.5847596003587,
 'MSE': 6743531.349681093,
 'RMSE': np.float64(2596.8310206251567),
 'MAPE': 6.299678830385583,
 'R2': 0.8388589038591459,
 'Bias': np.float64(190.08601934817094)}

# Additional Layers

In [64]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(
            64,
            return_sequences=True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.SimpleRNN(32),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [65]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.3790 - mae: 0.4713 - val_loss: 0.1725 - val_mae: 0.3224
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.2429 - mae: 0.3820 - val_loss: 0.1559 - val_mae: 0.3056
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.2152 - mae: 0.3587 - val_loss: 0.1402 - val_mae: 0.2897
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.1971 - mae: 0.3425 - val_loss: 0.1228 - val_mae: 0.2705
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.1838 - mae: 0.3306 - val_loss: 0.1232 - val_mae: 0.2697
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.1755 - mae: 0.3224 - val_loss: 0.1233 - val_mae: 0.2717
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.1682 - mae: 0.3157 - val_loss: 0.1110 - val_mae: 0.2560
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 35s 22ms/step - loss: 0.1629 - mae: 0.3104 - val_loss: 0.1205 - val_mae: 0.2688
Epoch 9/10
1584/1584 ━━━

{'MAE': 1924.4051489556705,
 'MSE': 6437859.547429205,
 'RMSE': np.float64(2537.2937448055172),
 'MAPE': 6.328667331927204,
 'R2': 0.8461631316768973,
 'Bias': np.float64(116.45219227880403)}

In [70]:
def build_rnn_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.SimpleRNN(
            128,
            return_sequences = True
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.SimpleRNN(64),

        tf.keras.layers.Dropout(0.5),
        
        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model

In [71]:
rnn_model_bn = build_rnn_model(
    sequence_length=sequence_length,
    n_features=19
)

rnn_model_bn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = rnn_model_bn.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

rnn_model_168 = rnn_model_bn.predict(
    X_test_seq
)

rnn_pred_original_168 = y_scaler.inverse_transform(
    rnn_model_168.reshape(-1, 1)
).reshape(rnn_model_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_168
)

results
    

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 49s 28ms/step - loss: 0.3098 - mae: 0.4189 - val_loss: 0.1246 - val_mae: 0.2723
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1700 - mae: 0.3189 - val_loss: 0.1031 - val_mae: 0.2438
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1495 - mae: 0.2980 - val_loss: 0.1029 - val_mae: 0.2461
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1380 - mae: 0.2859 - val_loss: 0.1021 - val_mae: 0.2490
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1292 - mae: 0.2763 - val_loss: 0.0961 - val_mae: 0.2366
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1236 - mae: 0.2698 - val_loss: 0.1012 - val_mae: 0.2445
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1184 - mae: 0.2642 - val_loss: 0.0863 - val_mae: 0.2230
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - loss: 0.1143 - mae: 0.2595 - val_loss: 0.0873 - val_mae: 0.2243
Epoch 9/10
1584/1584 ━━━

{'MAE': 1785.3066056561163,
 'MSE': 5417045.163471879,
 'RMSE': np.float64(2327.4546533653197),
 'MAPE': 6.006679776783465,
 'R2': 0.870556159640653,
 'Bias': np.float64(843.6400840578602)}

In [73]:
rnn_model_bn.save("/kaggle/working/rnn_model_168_87.keras")

# HyperParameter Tuning

In [75]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense
from tensorflow.keras.optimizers import Adam

def build_rnn_hyper(hp):

    model = Sequential()

    model.add(
            SimpleRNN(
                units=hp.Choice(
                    "lstm_units",
                    values=[32, 64, 128]
                )
            )        
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model
    
tuner_rnn = kt.RandomSearch(
    build_rnn_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_rnn.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64,
    verbose=1
)


Trial 5 Complete [00h 03m 30s]
val_loss: 0.08083296567201614

Best val_loss So Far: 0.07860761135816574
Total elapsed time: 00h 16m 57s


In [77]:

best_hp_rnn = tuner_rnn.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_rnn



In [78]:
best_rnn_model = tuner_rnn.get_best_models(
    num_models=1
)[0]

best_rnn_model.summary()



/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 16 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        18,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │           792 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,864 (93.22 KB)

 Trainable params: 23,864 (93.22 KB)

 Non-trainable params: 0 (0.00 B)

In [79]:
history_best_rnn = best_rnn_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.0410 - mae: 0.1491 - val_loss: 0.0863 - val_mae: 0.2130
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0398 - mae: 0.1472 - val_loss: 0.0866 - val_mae: 0.2176
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0385 - mae: 0.1449 - val_loss: 0.0790 - val_mae: 0.2005
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0376 - mae: 0.1435 - val_loss: 0.0815 - val_mae: 0.2060
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0365 - mae: 0.1414 - val_loss: 0.0794 - val_mae: 0.2020
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0357 - mae: 0.1400 - val_loss: 0.0834 - val_mae: 0.2085
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0350 - mae: 0.1387 - val_loss: 0.0862 - val_mae: 0.2127
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - loss: 0.0340 - mae: 0.1369 - val_loss: 0.0971 - val_mae: 0.2243
Epoch 9/10
1584/1584 ━━━

NameError: name 'best_bilstm_model' is not defined

In [80]:
rnn_pred_168_hy = best_rnn_model.predict(
    X_test_seq
)

rnn_pred_original_48 = y_scaler.inverse_transform(
    rnn_pred_168_hy.reshape(-1, 1)
).reshape(rnn_pred_168_hy.shape)

results = evaluate_deep_model(
    y_test_original_168,
    rnn_pred_original_48
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


{'MAE': 1983.0438260386416,
 'MSE': 7504264.136446263,
 'RMSE': np.float64(2739.391198139883),
 'MAPE': 6.473058432666447,
 'R2': 0.8206806959183726,
 'Bias': np.float64(733.4158961473513)}

In [6]:
data = [
    ["Feature Engineered", "Base", 2480.208866380491, 10927148.972322095, 3305.6238401128003, 8.410324175007107, 0.7388886220307018, 1228.078319740378],

    ["Dropout", "0.2", 1797.5445667555312, 5850402.645245208, 2418.7605597175607, 5.949787595291549, 0.8602007989234381, 565.7434255215156],
    ["Dropout", "0.35", 2255.6494169834923, 8691934.225433037, 2948.2086468621987, 7.450667563906278, 0.7923005416536363, 64.64306670061357],
    ["Dropout", "0.5", 1805.7853438086613, 5851812.077466108, 2419.051896397865, 5.883813948274826, 0.8601671196178584, -31.659633388577074],

    ["Batch Normalization", "Added", 2322.207466017461, 9269023.23703972, 3044.507059778269, 7.3228278024852465, 0.7785106219396067, -878.5953776645489],

    ["Optimizer", "SGD", 2458.7868039237246, 11013649.120632723, 3318.6818348001852, 7.559939461626539, 0.7368216443609413, -1251.5776093202937],
    ["Optimizer", "RMSprop", 2233.103843572446, 8596216.821249258, 2931.9305621465965, 7.556091483349981, 0.7945877716864085, 747.6869103180443],

    ["Learning Rate", "0.35", 1757.302142847611, 5535204.282548363, 2352.70148606838, 5.722750542844209, 0.8677326701394256, -109.67954021254077],
    ["Learning Rate", "0.5", 2169.64029475048, 8161735.031777124, 2856.875046580988, 7.250662821874354, 0.8049699984720955, 310.486800672894],

    ["Early Stopping", "Enabled", 2461.1576237533172, 11584741.290556297, 3403.636480377465, 8.039282919860984, 0.7231750231046621, -235.05391399176216],

    ["Batch Size", "32", 2328.7370715439965, 9147753.439265924, 3024.5253246197035, 7.818215398208486, 0.7814084431446604, 378.7625200680484],
    ["Batch Size", "16", 1925.5847596003587, 6743531.349681093, 2596.8310206251567, 6.299678830385583, 0.8388589038591459, 190.08601934817094],

    ["Additional Layers", "64, 32", 1924.4051489556705, 6437859.547429205, 2537.2937448055172, 6.328667331927204, 0.8461631316768973, 116.45219227880403],
    ["Additional Layers", "128, 64", 1785.3066056561163, 5417045.163471879, 2327.4546533653197, 6.006679776783465, 0.870556159640653, 843.6400840578602],

    ["Hyperparameter Tuning", "Tuned", 1983.0438260386416, 7504264.136446263, 2739.391198139883, 6.473058432666447, 0.8206806959183726, 733.4158961473513],
]

df_simplernn_168 = pd.DataFrame(
    data,
    columns=[
        "Experiment",
        "Configuration",
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Bias"
    ]
)

df_simplernn_168

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
0,Feature Engineered,Base,2480.208866,1.092715e+07,3305.623840,8.410324,0.738889,1228.078320
1,Dropout,0.2,1797.544567,5.850403e+06,2418.760560,5.949788,0.860201,565.743426
2,Dropout,0.35,2255.649417,8.691934e+06,2948.208647,7.450668,0.792301,64.643067
3,Dropout,0.5,1805.785344,5.851812e+06,2419.051896,5.883814,0.860167,-31.659633
4,Batch Normalization,Added,2322.207466,9.269023e+06,3044.507060,7.322828,0.778511,-878.595378
5,Optimizer,SGD,2458.786804,1.101365e+07,3318.681835,7.559939,0.736822,-1251.577609
6,Optimizer,RMSprop,2233.103844,8.596217e+06,2931.930562,7.556091,0.794588,747.686910
7,Learning Rate,0.35,1757.302143,5.535204e+06,2352.701486,5.722751,0.867733,-109.679540
8,Learning Rate,0.5,2169.640295,8.161735e+06,2856.875047,7.250663,0.804970,310.486801
9,Early Stopping,Enabled,2461.157624,1.158474e+07,3403.636480,8.039283,0.723175,-235.053914


In [7]:
df_simplernn_168.sort_values(by = "R2", ascending=False)

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
13,Additional Layers,"128, 64",1785.306606,5.417045e+06,2327.454653,6.006680,0.870556,843.640084
7,Learning Rate,0.35,1757.302143,5.535204e+06,2352.701486,5.722751,0.867733,-109.679540
1,Dropout,0.2,1797.544567,5.850403e+06,2418.760560,5.949788,0.860201,565.743426
3,Dropout,0.5,1805.785344,5.851812e+06,2419.051896,5.883814,0.860167,-31.659633
12,Additional Layers,"64, 32",1924.405149,6.437860e+06,2537.293745,6.328667,0.846163,116.452192
11,Batch Size,16,1925.584760,6.743531e+06,2596.831021,6.299679,0.838859,190.086019
14,Hyperparameter Tuning,Tuned,1983.043826,7.504264e+06,2739.391198,6.473058,0.820681,733.415896
8,Learning Rate,0.5,2169.640295,8.161735e+06,2856.875047,7.250663,0.804970,310.486801
6,Optimizer,RMSprop,2233.103844,8.596217e+06,2931.930562,7.556091,0.794588,747.686910
2,Dropout,0.35,2255.649417,8.691934e+06,2948.208647,7.450668,0.792301,64.643067


# Bi Lstm 168

In [22]:
sequence_length = 168
forecast_horizon = 24


X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    sequence_length,
    forecast_horizon
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    sequence_length,
    forecast_horizon
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    sequence_length,
    forecast_horizon
)

In [83]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [84]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0968 - mae: 0.2235 - val_loss: 0.0824 - val_mae: 0.2156
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0534 - mae: 0.1709 - val_loss: 0.0806 - val_mae: 0.2110
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0455 - mae: 0.1571 - val_loss: 0.0814 - val_mae: 0.2120
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0401 - mae: 0.1476 - val_loss: 0.0814 - val_mae: 0.2120
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0368 - mae: 0.1417 - val_loss: 0.0822 - val_mae: 0.2096
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0330 - mae: 0.1346 - val_loss: 0.0870 - val_mae: 0.2165
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0303 - mae: 0.1294 - val_loss: 0.0892 - val_mae: 0.2183
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step - loss: 0.0277 - mae: 0.1244 - val_loss: 0.0871 - val_mae: 0.2133
Epoch 9/10
1584/1584 ━━━

In [86]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step


{'MAE': 2162.8204846649555,
 'MSE': 8270395.9592299275,
 'RMSE': np.float64(2875.8296123431805),
 'MAPE': 7.1793800472871645,
 'R2': 0.8023734744775486,
 'Bias': np.float64(952.896434725676)}

# Drop out

In [23]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [24]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

I0000 00:00:1786593648.457135      85 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786593648.460341      85 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 31s 17ms/step - loss: 0.1251 - mae: 0.2620 - val_loss: 0.0878 - val_mae: 0.2224
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0748 - mae: 0.2074 - val_loss: 0.0772 - val_mae: 0.2085
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0659 - mae: 0.1940 - val_loss: 0.0741 - val_mae: 0.2023
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0611 - mae: 0.1866 - val_loss: 0.0748 - val_mae: 0.2053
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0576 - mae: 0.1813 - val_loss: 0.0750 - val_mae: 0.2040
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0548 - mae: 0.1769 - val_loss: 0.0820 - val_mae: 0.2119
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0521 - mae: 0.1726 - val_loss: 0.0740 - val_mae: 0.1995
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0499 - mae: 0.1692 - val_loss: 0.0703 - val_mae: 0.1933
Epoch 9/10
1584/1584 ━━━

In [32]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


{'MAE': 1828.0765311985845,
 'MSE': 6097819.193034872,
 'RMSE': np.float64(2469.376276114046),
 'MAPE': 6.025734703371503,
 'R2': 0.8542886185468913,
 'Bias': np.float64(462.629972559427)}

In [33]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [34]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 29s 17ms/step - loss: 0.1539 - mae: 0.2934 - val_loss: 0.0938 - val_mae: 0.2338
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0940 - mae: 0.2343 - val_loss: 0.0873 - val_mae: 0.2257
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0847 - mae: 0.2219 - val_loss: 0.0797 - val_mae: 0.2143
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0797 - mae: 0.2149 - val_loss: 0.0765 - val_mae: 0.2061
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0753 - mae: 0.2088 - val_loss: 0.0856 - val_mae: 0.2242
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0722 - mae: 0.2045 - val_loss: 0.0801 - val_mae: 0.2154
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0694 - mae: 0.2004 - val_loss: 0.0787 - val_mae: 0.2122
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0678 - mae: 0.1981 - val_loss: 0.0792 - val_mae: 0.2108
Epoch 9/10
1584/1584 ━━━

In [36]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


{'MAE': 1876.0153716699479,
 'MSE': 6298012.65059179,
 'RMSE': np.float64(2509.5841588980015),
 'MAPE': 6.190594843962213,
 'R2': 0.8495048648252638,
 'Bias': np.float64(318.7028055211679)}

# Batch Normalization

In [37]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),

        tf.keras.layers.BatchNormalization(),
        
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [38]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 32s 18ms/step - loss: 0.2474 - mae: 0.3652 - val_loss: 0.1098 - val_mae: 0.2514
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.1145 - mae: 0.2597 - val_loss: 0.0816 - val_mae: 0.2151
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0997 - mae: 0.2417 - val_loss: 0.0779 - val_mae: 0.2075
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 0.0919 - mae: 0.2318 - val_loss: 0.0821 - val_mae: 0.2161
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 29s 18ms/step - loss: 0.0861 - mae: 0.2242 - val_loss: 0.0788 - val_mae: 0.2137
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0826 - mae: 0.2193 - val_loss: 0.0741 - val_mae: 0.2025
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0789 - mae: 0.2146 - val_loss: 0.0814 - val_mae: 0.2115
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0761 - mae: 0.2108 - val_loss: 0.0867 - val_mae: 0.2197
Epoch 9/10
1584/1584 ━━━

In [39]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 1693.1953597906413,
 'MSE': 5488198.586627136,
 'RMSE': np.float64(2342.6904589866617),
 'MAPE': 5.462584954191552,
 'R2': 0.8688559020149574,
 'Bias': np.float64(128.79936751607013)}

# New OPtimizers

In [40]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64)
        ),
        
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [41]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 16ms/step - loss: 0.1519 - mae: 0.2924 - val_loss: 0.0975 - val_mae: 0.2364
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0984 - mae: 0.2392 - val_loss: 0.0836 - val_mae: 0.2186
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0892 - mae: 0.2273 - val_loss: 0.0818 - val_mae: 0.2134
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0836 - mae: 0.2198 - val_loss: 0.0741 - val_mae: 0.2031
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0795 - mae: 0.2143 - val_loss: 0.0821 - val_mae: 0.2163
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0763 - mae: 0.2101 - val_loss: 0.0828 - val_mae: 0.2213
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0739 - mae: 0.2066 - val_loss: 0.0806 - val_mae: 0.2101
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0713 - mae: 0.2029 - val_loss: 0.0846 - val_mae: 0.2252
Epoch 9/10
1584/1584 ━━━

In [42]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 1787.0569722029302,
 'MSE': 5701244.488513424,
 'RMSE': np.float64(2387.727892477161),
 'MAPE': 5.8503879349423675,
 'R2': 0.8637650307224414,
 'Bias': np.float64(277.3460541384742)}

In [44]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="sgd",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.5292 - mae: 0.5596 - val_loss: 0.2451 - val_mae: 0.3943
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.2714 - mae: 0.4104 - val_loss: 0.2008 - val_mae: 0.3554
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.2296 - mae: 0.3773 - val_loss: 0.1821 - val_mae: 0.3367
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.2044 - mae: 0.3549 - val_loss: 0.1729 - val_mae: 0.3263
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.1858 - mae: 0.3374 - val_loss: 0.1628 - val_mae: 0.3151
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.1717 - mae: 0.3232 - val_loss: 0.1597 - val_mae: 0.3109
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.1620 - mae: 0.3131 - val_loss: 0.1549 - val_mae: 0.3047
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 24s 15ms/step - loss: 0.1544 - mae: 0.3051 - val_loss: 0.1521 - val_mae: 0.3011
Epoch 9/10
1584/1584 ━━━

In [45]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 2484.933259451094,
 'MSE': 10291298.810632199,
 'RMSE': np.float64(3208.0054255927),
 'MAPE': 8.039697617911317,
 'R2': 0.7540826778929757,
 'Bias': np.float64(-596.4495050148081)}

# Changing Learning Rate

In [46]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.0005),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - loss: 0.1835 - mae: 0.3209 - val_loss: 0.1065 - val_mae: 0.2480
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.1025 - mae: 0.2451 - val_loss: 0.0836 - val_mae: 0.2186
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0904 - mae: 0.2298 - val_loss: 0.0819 - val_mae: 0.2178
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0844 - mae: 0.2218 - val_loss: 0.0773 - val_mae: 0.2097
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0802 - mae: 0.2161 - val_loss: 0.0800 - val_mae: 0.2120
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0772 - mae: 0.2117 - val_loss: 0.0779 - val_mae: 0.2097
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0751 - mae: 0.2087 - val_loss: 0.0778 - val_mae: 0.2119
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0725 - mae: 0.2052 - val_loss: 0.0804 - val_mae: 0.2139
Epoch 9/10
1584/1584 ━━━

In [47]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 1960.7312558593449,
 'MSE': 6692048.412549248,
 'RMSE': np.float64(2586.8993819917405),
 'MAPE': 6.52260622322505,
 'R2': 0.8400891223443566,
 'Bias': np.float64(508.45894961368026)}

In [48]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 30s 18ms/step - loss: 0.1531 - mae: 0.2925 - val_loss: 0.0935 - val_mae: 0.2322
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - loss: 0.0943 - mae: 0.2347 - val_loss: 0.0847 - val_mae: 0.2198
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 17ms/step - loss: 0.0851 - mae: 0.2225 - val_loss: 0.0807 - val_mae: 0.2138
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0798 - mae: 0.2151 - val_loss: 0.0754 - val_mae: 0.2066
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0758 - mae: 0.2096 - val_loss: 0.0785 - val_mae: 0.2086
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0726 - mae: 0.2052 - val_loss: 0.0742 - val_mae: 0.2031
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - loss: 0.0695 - mae: 0.2008 - val_loss: 0.0767 - val_mae: 0.2093
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0676 - mae: 0.1979 - val_loss: 0.0732 - val_mae: 0.1997
Epoch 9/10
1584/1584 ━━━

In [49]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 1758.1254042593173,
 'MSE': 5661104.840584852,
 'RMSE': np.float64(2379.3076389119697),
 'MAPE': 5.812292353634281,
 'R2': 0.8647241938864451,
 'Bias': np.float64(478.6013273940726)}

# Early Stopping

In [50]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [52]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    callbacks = early_stopping,
    epochs=20,
    batch_size=64
)

Epoch 1/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - loss: 0.1499 - mae: 0.2910 - val_loss: 0.1049 - val_mae: 0.2432
Epoch 2/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0971 - mae: 0.2373 - val_loss: 0.0846 - val_mae: 0.2173
Epoch 3/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 26s 16ms/step - loss: 0.0878 - mae: 0.2251 - val_loss: 0.0838 - val_mae: 0.2193
Epoch 4/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0820 - mae: 0.2175 - val_loss: 0.0758 - val_mae: 0.2052
Epoch 5/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0779 - mae: 0.2121 - val_loss: 0.0733 - val_mae: 0.2007
Epoch 6/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0748 - mae: 0.2077 - val_loss: 0.0799 - val_mae: 0.2144
Epoch 7/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0723 - mae: 0.2041 - val_loss: 0.0798 - val_mae: 0.2163
Epoch 8/20
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 25s 16ms/step - loss: 0.0702 - mae: 0.2013 - val_loss: 0.0781 - val_mae: 0.2102
Epoch 9/20
1584/1584 ━━━

In [54]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


{'MAE': 1716.292653290363,
 'MSE': 5244531.318543965,
 'RMSE': np.float64(2290.0941724182358),
 'MAPE': 5.66646222721273,
 'R2': 0.8746784908985142,
 'Bias': np.float64(216.42199049619254)}

# New Batch Size

In [55]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=32
)

Epoch 1/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 52s 16ms/step - loss: 0.1371 - mae: 0.2787 - val_loss: 0.0876 - val_mae: 0.2251
Epoch 2/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0927 - mae: 0.2316 - val_loss: 0.0784 - val_mae: 0.2136
Epoch 3/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0839 - mae: 0.2202 - val_loss: 0.0768 - val_mae: 0.2127
Epoch 4/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0784 - mae: 0.2128 - val_loss: 0.0769 - val_mae: 0.2059
Epoch 5/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0746 - mae: 0.2076 - val_loss: 0.0717 - val_mae: 0.1993
Epoch 6/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 51s 16ms/step - loss: 0.0716 - mae: 0.2033 - val_loss: 0.0801 - val_mae: 0.2141
Epoch 7/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 50s 16ms/step - loss: 0.0687 - mae: 0.1993 - val_loss: 0.0837 - val_mae: 0.2180
Epoch 8/10
3167/3167 ━━━━━━━━━━━━━━━━━━━━ 49s 16ms/step - loss: 0.0671 - mae: 0.1967 - val_loss: 0.0752 - val_mae: 0.2041
Epoch 9/10
3167/3167 ━━━

In [57]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


{'MAE': 1814.0278454738534,
 'MSE': 5866288.482441753,
 'RMSE': np.float64(2422.0422131832784),
 'MAPE': 5.9460587534860645,
 'R2': 0.8598211964442286,
 'Bias': np.float64(207.18426348752791)}

In [59]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=16
)

Epoch 1/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 97s 15ms/step - loss: 0.1288 - mae: 0.2704 - val_loss: 0.0897 - val_mae: 0.2256
Epoch 2/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0904 - mae: 0.2286 - val_loss: 0.0820 - val_mae: 0.2189
Epoch 3/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0820 - mae: 0.2171 - val_loss: 0.0738 - val_mae: 0.2018
Epoch 4/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0767 - mae: 0.2099 - val_loss: 0.0775 - val_mae: 0.2077
Epoch 5/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0721 - mae: 0.2036 - val_loss: 0.0757 - val_mae: 0.2071
Epoch 6/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 93s 15ms/step - loss: 0.0686 - mae: 0.1987 - val_loss: 0.0745 - val_mae: 0.2041
Epoch 7/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0658 - mae: 0.1946 - val_loss: 0.0878 - val_mae: 0.2281
Epoch 8/10
6334/6334 ━━━━━━━━━━━━━━━━━━━━ 94s 15ms/step - loss: 0.0634 - mae: 0.1911 - val_loss: 0.0847 - val_mae: 0.2164
Epoch 9/10
6334/6334 ━━━

In [60]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


{'MAE': 1883.608025565643,
 'MSE': 6598037.902321186,
 'RMSE': np.float64(2568.664614604481),
 'MAPE': 6.1030707441361915,
 'R2': 0.8423355650286672,
 'Bias': np.float64(-232.75499967106757)}

# Additional Layers

In [63]:
def build_bilstm_model(sequence_length, n_features):

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, n_features)
        ),

        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(
                64,
                return_sequences = True
            )
        ),

        tf.keras.layers.BatchNormalization(),
        
        tf.keras.layers.Dropout(0.5),

         tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(32)
        ),

        tf.keras.layers.BatchNormalization(),
        
        tf.keras.layers.Dropout(0.5),

        tf.keras.layers.Dense(
            64,
            activation="relu"
        ),

        tf.keras.layers.Dense(24)
    ])

    return model



In [64]:
bilstm_model_168 = build_bilstm_model(
    sequence_length=sequence_length,
    n_features=19
)

bilstm_model_168.compile(
    optimizer="rmsprop",
    loss="mse",
    metrics=["mae"]
)

history = bilstm_model_168.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 52s 31ms/step - loss: 0.2622 - mae: 0.3826 - val_loss: 0.1169 - val_mae: 0.2626
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 47s 30ms/step - loss: 0.1473 - mae: 0.2959 - val_loss: 0.0938 - val_mae: 0.2336
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.1295 - mae: 0.2767 - val_loss: 0.1018 - val_mae: 0.2482
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.1166 - mae: 0.2622 - val_loss: 0.1115 - val_mae: 0.2549
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.1084 - mae: 0.2528 - val_loss: 0.0877 - val_mae: 0.2258
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.1015 - mae: 0.2447 - val_loss: 0.0921 - val_mae: 0.2344
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.0961 - mae: 0.2382 - val_loss: 0.1018 - val_mae: 0.2484
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 0.0926 - mae: 0.2336 - val_loss: 0.0969 - val_mae: 0.2401
Epoch 9/10
1584/1584 ━━━

In [65]:
bilstm_pred_168 = bilstm_model_168.predict(
    X_test_seq
)

bilstm_pred_original_168 = y_scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_168
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step


{'MAE': 1946.0412458739283,
 'MSE': 6385312.478979257,
 'RMSE': np.float64(2526.9175845245245),
 'MAPE': 6.469059109147861,
 'R2': 0.8474187782765699,
 'Bias': np.float64(442.83422582483996)}

# Hyperparameter for 168

In [67]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense
from tensorflow.keras.optimizers import Adam

def build_bilstm_hyper(hp):

    model = Sequential()

    model.add(
        Bidirectional(
            LSTM(
                units=hp.Choice(
                    "lstm_units",
                    values=[32, 64, 128]
                )
            ),
            input_shape=(
                X_train_seq.shape[1],
                X_train_seq.shape[2]
            )
        )
    )

    model.add(
        Dense(
            units=hp.Choice(
                "dense_units",
                values=[32, 64, 128]
            ),
            activation="relu"
        )
    )

    # Next 24 hours
    model.add(Dense(24))

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.0001, 0.0005, 0.001]
    )

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate
        ),
        loss="mse",
        metrics=["mae"]
    )

    return model
tuner_bilstm = kt.RandomSearch(
    build_bilstm_hyper,
    objective="val_loss",
    max_trials=5,
    directory="hyperparameter_tuning",
    project_name="bilstm_48_to_24"
)


tuner_bilstm.search(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64,
    verbose=1
)



Trial 5 Complete [00h 04m 32s]
val_loss: 0.0721234604716301

Best val_loss So Far: 0.06961983442306519
Total elapsed time: 00h 23m 15s


In [68]:
best_hp_bilstm = tuner_bilstm.get_best_hyperparameters(
    num_trials=1
)[0]

best_hp_bilstm

In [69]:
best_bilstm_model = tuner_bilstm.get_best_models(
    num_models=1
)[0]

best_bilstm_model.summary()



/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 256)            │       151,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 169,560 (662.34 KB)

 Trainable params: 169,560 (662.34 KB)

 Non-trainable params: 0 (0.00 B)

In [70]:
history_best_bilstm = best_bilstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=10,
    batch_size=64
)



Epoch 1/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - loss: 0.0397 - mae: 0.1467 - val_loss: 0.0690 - val_mae: 0.1897
Epoch 2/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0316 - mae: 0.1318 - val_loss: 0.0742 - val_mae: 0.1967
Epoch 3/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0260 - mae: 0.1205 - val_loss: 0.0788 - val_mae: 0.2033
Epoch 4/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0226 - mae: 0.1129 - val_loss: 0.0771 - val_mae: 0.2029
Epoch 5/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0186 - mae: 0.1027 - val_loss: 0.0780 - val_mae: 0.2027
Epoch 6/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - loss: 0.0163 - mae: 0.0965 - val_loss: 0.0736 - val_mae: 0.1977
Epoch 7/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - loss: 0.0143 - mae: 0.0905 - val_loss: 0.0755 - val_mae: 0.1986
Epoch 8/10
1584/1584 ━━━━━━━━━━━━━━━━━━━━ 28s 17ms/step - loss: 0.0128 - mae: 0.0856 - val_loss: 0.0800 - val_mae: 0.2083
Epoch 9/10
1584/1584 ━━━

In [72]:
bilstm_pred_48_hy = best_bilstm_model.predict(
    X_test_seq
)

bilstm_pred_original_48 = y_scaler.inverse_transform(
    bilstm_pred_48_hy.reshape(-1, 1)
).reshape(bilstm_pred_48_hy.shape)

results = evaluate_deep_model(
    y_test_original_168,
    bilstm_pred_original_48
)

results

674/674 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step


{'MAE': 1698.8071989270431,
 'MSE': 5450156.782722099,
 'RMSE': np.float64(2334.557084913988),
 'MAPE': 5.514244585228028,
 'R2': 0.8697649358227002,
 'Bias': np.float64(287.0695592235975)}

In [8]:
data = [
    ["Feature Engineered", "Base", 2162.8204846649555, 8270395.9592299275, 2875.8296123431805, 7.1793800472871645, 0.8023734744775486, 952.896434725676],

    ["Dropout", "0.35", 1828.0765311985845, 6097819.193034872, 2469.376276114046, 6.025734703371503, 0.8542886185468913, 462.629972559427],
    ["Dropout", "0.5", 1876.0153716699479, 6298012.65059179, 2509.5841588980015, 6.190594843962213, 0.8495048648252638, 318.7028055211679],

    ["Batch Normalization", "Added", 1693.1953597906413, 5488198.586627136, 2342.6904589866617, 5.462584954191552, 0.8688559020149574, 128.79936751607013],

    ["Optimizer", "SGD", 2484.933259451094, 10291298.810632199, 3208.0054255927, 8.039697617911317, 0.7540826778929757, -596.4495050148081],
    ["Optimizer", "RMSprop", 1787.0569722029302, 5701244.488513424, 2387.727892477161, 5.8503879349423675, 0.8637650307224414, 277.3460541384742],

    ["Learning Rate", "0.0005", 1960.7312558593449, 6692048.412549248, 2586.8993819917405, 6.52260622322505, 0.8400891223443566, 508.45894961368026],
    ["Learning Rate", "0.001", 1758.1254042593173, 5661104.840584852, 2379.3076389119697, 5.812292353634281, 0.8647241938864451, 478.6013273940726],

    ["Early Stopping", "Enabled", 1716.292653290363, 5244531.318543965, 2290.0941724182358, 5.66646222721273, 0.8746784908985142, 216.42199049619254],

    ["Batch Size", "32", 1814.0278454738534, 5866288.482441753, 2422.0422131832784, 5.9460587534860645, 0.8598211964442286, 207.18426348752791],
    ["Batch Size", "16", 1883.608025565643, 6598037.902321186, 2568.664614604481, 6.1030707441361915, 0.8423355650286672, -232.75499967106757],

    ["Additional Layers", "Added", 1946.0412458739283, 6385312.478979257, 2526.9175845245245, 6.469059109147861, 0.8474187782765699, 442.83422582483996],

    ["Hyperparameter Tuning", "Tuned", 1698.8071989270431, 5450156.782722099, 2334.557084913988, 5.514244585228028, 0.8697649358227002, 287.0695592235975],
]

df_bilstm_168 = pd.DataFrame(
    data,
    columns=[
        "Experiment",
        "Configuration",
        "MAE",
        "MSE",
        "RMSE",
        "MAPE",
        "R2",
        "Bias"
    ]
)

df_bilstm_168

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
0,Feature Engineered,Base,2162.820485,8.270396e+06,2875.829612,7.179380,0.802373,952.896435
1,Dropout,0.35,1828.076531,6.097819e+06,2469.376276,6.025735,0.854289,462.629973
2,Dropout,0.5,1876.015372,6.298013e+06,2509.584159,6.190595,0.849505,318.702806
3,Batch Normalization,Added,1693.195360,5.488199e+06,2342.690459,5.462585,0.868856,128.799368
4,Optimizer,SGD,2484.933259,1.029130e+07,3208.005426,8.039698,0.754083,-596.449505
5,Optimizer,RMSprop,1787.056972,5.701244e+06,2387.727892,5.850388,0.863765,277.346054
6,Learning Rate,0.0005,1960.731256,6.692048e+06,2586.899382,6.522606,0.840089,508.458950
7,Learning Rate,0.001,1758.125404,5.661105e+06,2379.307639,5.812292,0.864724,478.601327
8,Early Stopping,Enabled,1716.292653,5.244531e+06,2290.094172,5.666462,0.874678,216.421990
9,Batch Size,32,1814.027845,5.866288e+06,2422.042213,5.946059,0.859821,207.184263


In [9]:
df_bilstm_168.sort_values(by = "R2", ascending=False)

,Experiment,Configuration,MAE,MSE,RMSE,MAPE,R2,Bias
8,Early Stopping,Enabled,1716.292653,5.244531e+06,2290.094172,5.666462,0.874678,216.421990
12,Hyperparameter Tuning,Tuned,1698.807199,5.450157e+06,2334.557085,5.514245,0.869765,287.069559
3,Batch Normalization,Added,1693.195360,5.488199e+06,2342.690459,5.462585,0.868856,128.799368
7,Learning Rate,0.001,1758.125404,5.661105e+06,2379.307639,5.812292,0.864724,478.601327
5,Optimizer,RMSprop,1787.056972,5.701244e+06,2387.727892,5.850388,0.863765,277.346054
9,Batch Size,32,1814.027845,5.866288e+06,2422.042213,5.946059,0.859821,207.184263
1,Dropout,0.35,1828.076531,6.097819e+06,2469.376276,6.025735,0.854289,462.629973
2,Dropout,0.5,1876.015372,6.298013e+06,2509.584159,6.190595,0.849505,318.702806
11,Additional Layers,Added,1946.041246,6.385312e+06,2526.917585,6.469059,0.847419,442.834226
10,Batch Size,16,1883.608026,6.598038e+06,2568.664615,6.103071,0.842336,-232.755000
